In [5]:
# une seule tetrode + raster + EEG brut + bande-son + t_spk réinjectés dans EEG. Curseur et apparition au centre
# adapté aux stims.
# export dans le dossier patient

import os
import wave
import subprocess
from pathlib import Path
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.collections import LineCollection
from matplotlib.patches import Rectangle
import pynapple as nap
from pynapple.io.interface_nwb import NWBFile


# =========================
# Paramètres
# =========================
patient = "P127_DS76" #"P111_SL64"
sess = "5"
tetrode_id = 10.0               # 1.0 ... 14.0 (float)
root_data = "/media/aube/Aube2/" # "/home/aube/Documents/article_neuronal_stimic/raster_video/"

# Stimulation
# PRE_SEC = 3.0          # secondes avant la stim (temps réel)
# POST_SEC = 3.0         # secondes après la stim (temps réel)

# PRE affiché, relatif au début de stimulation
PRE_REL_START = -4.0   # début du pre : stim_t0 - 4 s
PRE_REL_END   = -1.0   # fin du pre : stim_t0 - 1 s

POST_SEC = 3.0  # POST affiché, forcément collé à la fin de stimulation

STIM_DISPLAY = 0.2     # durée affichée de la stimulation (en secondes vidéo)
STIM_INDEX = 2 # 7         # quelle stimulation dans le fichier (0 = première)

# Vidéo
window = 1 # secondes (x-axis: -window => + window)
speed = 0.2 # 0.3 s pour window de 4 s 
fps = 30

# Raster style
LINE_W = 1.6
HALF_H = 0.45

# Brut .dat
FS = 32768.0
DTYPE = np.int16

# Affichage brut
DARK = True
BG = "#0b0f14"  if DARK else "white" # noir bleuté (moins dur que #000)
FG = "#c7d0d9" if DARK else "black" # texte/grille doux
RAW_COLOR = "#545f69" #FG          # même couleur pour tous les canaux
RAW_ALPHA = 0.95
GRID_ALPHA = 0.10
RAW_LINE_W = 0.2
RAW_DOWNSAMPLE_TARGET = 10000   # points max affichés par frame (perfs)

# autoscale du signal
RAW_VISUAL_GAIN = 2.0  # gain d'affichage (multiplie le signal/l'amplitude)
RAW_CLIP = 4.0
Y_MARGIN = 0.8
ROBUST_PCT = 99.0       # percentile pour échelle
RAW_OFFSET_MULT = 2.5    # offset vertical entre canaux (en unités int16)

# Audio
SR = 44100
CLICK_DUR = 0.020
BASE_FREQ = 450.0
FREQ_STEP = 70.0

# =========================
# Load NWB et stims
# =========================
def get_nwb(patient, session, root=""): # root="/media/aube/Aube/"
    path_folder = root + "Spike-sorting/Data_folders/" + patient + "/" + patient + "_stim" + session + "/"
    files_basename = patient + "_stim" + session
    nwbfile_path = path_folder + files_basename + ".nwb"
    print("NWB path:", nwbfile_path)

    if not os.path.exists(nwbfile_path):
        from datetime import datetime
        from dateutil import tz
        from neuroconv.datainterfaces import NeuroScopeSortingInterface

        xml_path = Path(path_folder + files_basename + ".xml")
        interface = NeuroScopeSortingInterface(folder_path=Path(path_folder), xml_file_path=xml_path, verbose=False)
        metadata = interface.get_metadata()
        session_start_time = datetime(2023, 4, 4, 12, 30, 0, tzinfo=tz.gettz("US/Pacific"))
        metadata["NWBFile"].update(session_start_time=session_start_time)
        interface.run_conversion(nwbfile_path=nwbfile_path, metadata=metadata)

    return NWBFile(nwbfile_path)["units"], Path(nwbfile_path)

def get_stim(patient, session, root="/home/aube/Documents/article_neuronal_stimic/raster_video/"): # root="/media/aube/Aube/"
    stim_path = root + "Spike-sorting/Data_folders/" + patient + "/" + patient + "_stim" + session + "/" + f"{patient}_stim{sess}_stim_events_TRC_re-shifted_loca.txt"
    if not os.path.exists(stim_path):
        raise FileNotFoundError(f"Fichier stim introuvable: {stim_path}")

    # adapte le sep si nécessaire (tab/espaces)
    stim_df = pd.read_csv(stim_path, sep=", ", engine="python", header=None)
    print(stim_df)
    stim_df.columns = ["label", "t_start", "dur", "loca"]

    # sélection
    if STIM_INDEX < 0 or STIM_INDEX >= len(stim_df):
        raise ValueError(f"STIM_INDEX invalide ({STIM_INDEX}) — nb stims={len(stim_df)}")

    stim_label = stim_df.loc[STIM_INDEX, "label"]
    stim_t0 = float(stim_df.loc[STIM_INDEX, "t_start"])
    stim_dur = float(stim_df.loc[STIM_INDEX, "dur"])
    stim_t1 = stim_t0 + stim_dur

    print(f"Stim choisie: idx={STIM_INDEX} label={stim_label} t0={stim_t0:.3f}s dur={stim_dur:.3f}s")
    return [stim_label, stim_t0, stim_dur, stim_t1]

# =========================
# Outils brut .dat (memmap)
# =========================
def open_dat_memmap(dat_path: Path, n_channels: int, dtype=np.int16):
    # Attention : marche que si tous les canaux sont inclus dans le .dat
    if not dat_path.exists():
        raise FileNotFoundError(f".dat introuvable: {dat_path.resolve()}")

    bytes_per_val = np.dtype(dtype).itemsize
    n_vals = dat_path.stat().st_size // bytes_per_val
    if n_vals % n_channels != 0:
        raise ValueError(
            f"Taille .dat incompatible: n_vals={n_vals} n'est pas multiple de n_channels={n_channels}."
        )
    n_samples = n_vals // n_channels
    mm = np.memmap(dat_path, dtype=dtype, mode="r", shape=(n_samples, n_channels), order="C")
    return mm, n_samples

def tetrode_channels(tetrode_id):
    # tetrode_id peut être float -> cast int
    tt = int(round(float(tetrode_id)))
    if tt < 1 or tt > N_TETRODES:
        raise ValueError(f"tetrode_id={tetrode_id} invalide. Attendu: 1..{N_TETRODES}.")
    start = (tt - 1) * 4
    return list(range(start, start + 4))

def get_raw_window(mm, fs, t0, t1, channels, t_min_valid=0.0, target_points=2500):
    """
    Fenêtre [t0, t1] en temps ABSOLU, avec padding NaN si t0 < t_min_valid.
    Retourne x_rel in [-window, 0], seg shape (n_points, n_ch) en float32.
    """
    n_samples_total = mm.shape[0]
    n_ch = len(channels)

    # indices en samples (peuvent être <0)
    s0 = int(np.floor(t0 * fs))
    s1 = int(np.floor(t1 * fs))

    s_min = int(np.floor(t_min_valid * fs))

    # longueur cible en samples (fenêtre constante)
    n_target = max(1, int(np.round((t1 - t0) * fs)))

    # padding à gauche si on est avant t_min_valid
    left_pad = max(0, s_min - s0)

    # clamp lecture
    s0_read = max(s_min, s0)
    s1_read = min(n_samples_total, s1)

    if s1_read > s0_read:
        seg = mm[s0_read:s1_read, channels].astype(np.float32)
    else:
        seg = np.empty((0, n_ch), dtype=np.float32)

    # ajoute padding NaN à gauche
    if left_pad > 0:
        seg = np.vstack([np.full((left_pad, n_ch), np.nan, dtype=np.float32), seg])

    # ajuste à n_target (pad NaN à droite si besoin)
    if seg.shape[0] > n_target:
        seg = seg[:n_target]
    elif seg.shape[0] < n_target:
        pad = n_target - seg.shape[0]
        seg = np.vstack([seg, np.full((pad, n_ch), np.nan, dtype=np.float32)])

    # downsample pour affichage
    n = seg.shape[0]
    if n > target_points:
        step = int(np.ceil(n / target_points))
        seg = seg[::step, :]
        # x_rel recalculé avec step
        x_rel = (np.arange(seg.shape[0], dtype=np.float32) * (step / fs)) - (t1 - t0)
    else:
        x_rel = (np.arange(n, dtype=np.float32) / fs) - (t1 - t0)

    return x_rel, seg

# =========================
# Audio (morse)
# =========================
def synth_morse_audio_from_spike_u(spike_u, U_TOTAL, speed, sr=44100,
                                   click_dur=0.020, base_freq=450.0, freq_step=70.0):
    duration_video = float(U_TOTAL) / float(speed)
    n = int(np.ceil(duration_video * sr))
    audio = np.zeros(n, dtype=np.float32)

    L = max(1, int(click_dur * sr))
    tt = np.arange(L, dtype=np.float32) / sr

    env = np.exp(-tt * 45.0).astype(np.float32)
    attack = np.minimum(1.0, tt / 0.002).astype(np.float32)
    env *= attack

    for i, su in enumerate(spike_u):
        if su.size == 0:
            continue
        f = base_freq + i * freq_step
        tone = (np.sin(2 * np.pi * f * tt) * env).astype(np.float32)

        for u_spk in su:
            # u_spk est en [0, U_TOTAL]
            idx0 = int((float(u_spk) / speed) * sr)
            if idx0 < 0 or idx0 >= n:
                continue
            idx1 = min(n, idx0 + L)
            audio[idx0:idx1] += tone[: idx1 - idx0]

    peak = float(np.max(np.abs(audio))) if audio.size else 0.0
    if peak > 0:
        audio /= (peak * 1.05)
    return audio


def write_wav(path, audio, sr=44100):
    audio_i16 = np.int16(np.clip(audio, -1.0, 1.0) * 32767)
    with wave.open(str(path), "wb") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(sr)
        wf.writeframes(audio_i16.tobytes())

def mux_audio_video_ffmpeg(video_in, audio_in, video_out):
    cmd = [
        "ffmpeg", "-y",
        "-i", str(video_in),
        "-i", str(audio_in),
        "-c:v", "copy",
        "-c:a", "aac",
        "-shortest",
        str(video_out),
    ]
    subprocess.run(cmd, check=True)


# =========================
# 1) Charger stim, et fonctions de warp (réel ↔ vidéo)
# =========================
stim_carac = get_stim(patient, sess, root_data)
stim_label, stim_t0, stim_dur, stim_t1 = stim_carac[0], stim_carac[1], stim_carac[2], stim_carac[3]

pre_t0 = stim_t0 + PRE_REL_START
pre_t1 = stim_t0 + PRE_REL_END
post_t0 = stim_t1
post_t1 = stim_t1 + POST_SEC

t_start = pre_t0
t_end = post_t1

# t_start = stim_t0 - PRE_SEC
# t_end   = stim_t1 + POST_end

elec_plot = stim_label[:stim_label.index('mA')-3]
elec = re.match(r"([A-Za-z'_]+)(\d+)-([A-Za-z'_]+)(\d+)", elec_plot).group(1).replace('_','')
plots = re.match(r"([A-Za-z'_]+)(\d+)-([A-Za-z'_]+)(\d+)", elec_plot).group(2) +'-'+ re.match(r"([A-Za-z'_]+)(\d+)-([A-Za-z'_]+)(\d+)", elec_plot).group(4)
freq = stim_label[stim_label.index('mA')+2:stim_label.index('Hz')-2]+' '+stim_label[stim_label.index('Hz'):stim_label.index('Hz')+2]
intensity = stim_label[stim_label.index('mA')-3:stim_label.index('mA')]+' '+stim_label[stim_label.index('mA'):stim_label.index('mA')+2]

# temps "vidéo" u=0 au début de la fenêtre réelle
r0 = t_start
r_pre_end = stim_t0
r_stim_end = stim_t1
r1 = t_end

# U_PRE = PRE_SEC
# U_STIM = STIM_DISPLAY
# U_POST = POST_SEC
# U_TOTAL = U_PRE + U_STIM + U_POST

# stim_u0 = U_PRE
# stim_u1 = U_PRE + U_STIM

# def warp(t):
#     """t (sec réel) -> u (sec vidéo), piecewise linéaire."""
#     t = np.asarray(t, dtype=np.float64)
#     u = np.empty_like(t)

#     # pre: slope 1
#     m1 = (t <= r_pre_end)
#     u[m1] = (t[m1] - r0)

#     # stim: compressed
#     m2 = (t > r_pre_end) & (t <= r_stim_end)
#     if stim_dur <= 0:
#         raise ValueError("durée stim <= 0 dans le fichier")
#     u[m2] = U_PRE + (t[m2] - r_pre_end) * (U_STIM / stim_dur)

#     # post: slope 1
#     m3 = (t > r_stim_end)
#     u[m3] = (U_PRE + U_STIM) + (t[m3] - r_stim_end)

#     return u

# def unwarp(u):
#     """u (sec vidéo) -> t (sec réel). (si tu en as besoin)"""
#     u = np.asarray(u, dtype=np.float64)
#     t = np.empty_like(u)

#     m1 = (u <= U_PRE)
#     t[m1] = r0 + u[m1]

#     m2 = (u > U_PRE) & (u <= U_PRE + U_STIM)
#     t[m2] = r_pre_end + (u[m2] - U_PRE) * (stim_dur / U_STIM)

#     m3 = (u > U_PRE + U_STIM)
#     t[m3] = r_stim_end + (u[m3] - (U_PRE + U_STIM))

#     return t
U_PRE = pre_t1 - pre_t0
U_STIM = STIM_DISPLAY
U_POST = POST_SEC
U_TOTAL = U_PRE + U_STIM + U_POST

stim_u0 = U_PRE
stim_u1 = U_PRE + U_STIM
post_u0 = stim_u1

def real_to_u_pre(t):
    return t - pre_t0

def real_to_u_post(t):
    return post_u0 + (t - post_t0)

def u_to_real(u):
    """
    Pour le timecode uniquement.
    Pendant le patch stim, on renvoie un temps interpolé dans la stimulation réelle.
    """
    u = np.asarray(u, dtype=float)
    t = np.empty_like(u)

    m_pre = u < stim_u0
    m_stim = (u >= stim_u0) & (u < stim_u1)
    m_post = u >= stim_u1

    t[m_pre] = pre_t0 + u[m_pre]

    # interpolation visuelle dans la stimulation réelle
    if U_STIM > 0:
        t[m_stim] = stim_t0 + ((u[m_stim] - stim_u0) / U_STIM) * stim_dur
    else:
        t[m_stim] = stim_t0

    t[m_post] = post_t0 + (u[m_post] - post_u0)

    return t


# ==================================
# 1) Charger spikes, filtrer tétrode
# ==================================
spk, nwb_path = get_nwb(patient, sess, root_data)

epoch_scroll = nap.IntervalSet(start=[t_start], end=[t_end], time_units="s")
spk = spk.restrict(epoch_scroll)

groups_dict = spk.getby_category("group")
available = sorted([float(g) for g in groups_dict.keys()])
if float(tetrode_id) not in [float(g) for g in groups_dict.keys()]:
    raise ValueError(
        f"Aucune unit pour tetrode/group={tetrode_id}. Groupes dispo (intervalle): {available}"
    )

idx_keep = np.asarray(groups_dict[float(tetrode_id)].index, dtype=int)
spk = spk[idx_keep]
unit_keys = list(spk.index)

spike_times = []
for k in unit_keys:
    spike_times.append(np.sort(np.asarray(spk[k].t, dtype=float)))

n_units = len(spike_times)
print(f"Tétrode {tetrode_id} | units: {n_units} | keys(ex): {unit_keys[:8]}")

# Couleurs par unit
if n_units < 12:
    cmap_units = plt.get_cmap("Set3")  # très lisible sur fond sombre #plt.get_cmap("tab20")
    unit_colors = [cmap_units(i % cmap_units.N) for i in range(n_units)]
else:
    import matplotlib.colors as mcolors
    def lighten(color, amount=0.55): # on éclaircit une colormap plus foncée (0.55 trop éclairci)
        r, g, b, a = mcolors.to_rgba(color)
        r = r + (1 - r) * amount
        g = g + (1 - g) * amount
        b = b + (1 - b) * amount
        return (r, g, b, a)

    cmap_units = plt.get_cmap("tab20")
    unit_colors = [lighten(cmap_units(i % cmap_units.N), amount=0.2) for i in range(n_units)]


# =========================
# 2) Ouvrir brut .dat
# =========================
dat_path = nwb_path.with_suffix(".dat")
N_TETRODES = pd.read_csv(root_data + "Spike-sorting/Data_folders/" + patient + "/mapping_anat_" + patient + '.txt').shape[0]
N_CHANNELS = N_TETRODES * 4 
mm, n_samples = open_dat_memmap(dat_path, n_channels=N_CHANNELS, dtype=DTYPE) # matrice dat,  nb de samples d'un canal
dur_dat = n_samples / FS
print(f"DAT: {dat_path.name} | n_samples={n_samples} | durée~{dur_dat:.2f}s | n_channels={N_CHANNELS}")

# Vérif temps
t_start = max(0.0, float(t_start))
t_end = min(float(t_end), float(dur_dat))
if t_end <= t_start:
    raise ValueError(f"t_end ({t_end}) doit être > t_start ({t_start}).")

# Canaux de la tétrode
raw_ch = tetrode_channels(tetrode_id)
print(f"Canaux bruts tétrode {tetrode_id}: {raw_ch}")

def precompute_minmax_envelope(mm, fs, t_start, t_end, channels, ds_fs=1000):
    """
    Produit une enveloppe min/max à fréquence ds_fs (Hz) sur [t_start, t_end].
    Retour:
      t_ds (sec absolues), ymin (n_pts, n_ch), ymax (n_pts, n_ch)
    """
    s0 = int(np.floor(t_start * fs))
    s1 = int(np.floor(t_end * fs))
    s0 = max(0, s0); s1 = min(mm.shape[0], s1)
    if s1 <= s0:
        raise ValueError("Intervalle brut vide (vérifie t_start/t_end).")

    # taille de bloc en samples
    block = max(1, int(round(fs / ds_fs)))
    n = s1 - s0
    n_blocks = n // block

    # on tronque à un multiple de block pour reshape
    n_use = n_blocks * block
    X = mm[s0:s0+n_use, channels].astype(np.float32)  # (n_use, n_ch)
    X = X.reshape(n_blocks, block, len(channels))

    ymin = np.min(X, axis=1)
    ymax = np.max(X, axis=1)
    # temps au centre de chaque bloc
    t_ds = t_start + (np.arange(n_blocks, dtype=np.float32) + 0.5) * (block / fs)
    return t_ds, ymin, ymax, block

# ex: downsampling a 1000 Hz d'affichage (ok aussi pour 500 ou 2000)
DS_FS = 2000
HILITE_MS = 6
HILITE_HALF_PTS = max(1, int(round(HILITE_MS * DS_FS / 1000.0)))

# t_ds, y_min, y_max, block = precompute_minmax_envelope(mm, FS, t_start, t_end, raw_ch, ds_fs=DS_FS)
t_ds_pre, y_min_pre, y_max_pre, block_pre = precompute_minmax_envelope(mm, FS, pre_t0, pre_t1, raw_ch, ds_fs=DS_FS)
t_ds_post, y_min_post, y_max_post, block_post = precompute_minmax_envelope(mm, FS, post_t0, post_t1, raw_ch, ds_fs=DS_FS)

u_ds_pre = real_to_u_pre(t_ds_pre)
u_ds_post = real_to_u_post(t_ds_post)

u_ds = np.concatenate([u_ds_pre, u_ds_post])
y_min = np.vstack([y_min_pre, y_min_post])
y_max = np.vstack([y_max_pre, y_max_post])

print(f"RAW downsample précomputé: {DS_FS} Hz | points={len(t_ds_pre)+len(t_ds_post)} | block={block_pre+block_post} samples")

# Conversion du signal et des spikes en temps warpé :
# u_ds = warp(t_ds)  # même longueur que t_ds, monotone
# spike_u = [warp(st) for st in spike_times]  # liste d'arrays en temps vidéo
spike_u = []

for st in spike_times:
    st_pre = st[(st >= pre_t0) & (st <= pre_t1)]
    st_post = st[(st >= post_t0) & (st <= post_t1)]

    u_pre = real_to_u_pre(st_pre)
    u_post = real_to_u_post(st_post)

    spike_u.append(np.sort(np.concatenate([u_pre, u_post])))

# ---- gain/scale RAW fixe sur tout l'intervalle (stable) basé sur signal pré-stim ----
def estimate_global_scale(mm, fs, t_start, t_end, channels, n_probe=80_000, pct=99.0):
    # on prend quelques samples répartis uniformément sur l'intervalle
    s0 = int(t_start * fs)
    s1 = int(t_end * fs)
    s0 = max(0, s0); s1 = min(mm.shape[0], s1)
    if s1 <= s0:
        return 1.0
    n_total = s1 - s0
    step = max(1, n_total // n_probe)
    seg = mm[s0:s1:step, channels].astype(np.float32)
    val = np.nanpercentile(np.abs(seg), pct)
    if not np.isfinite(val) or val <= 0:
        val = 1.0
    return float(val)

S_RAW = estimate_global_scale(mm, FS, pre_t0, pre_t1, raw_ch, n_probe=120_000, pct=ROBUST_PCT)
print(f"RAW scale (p{ROBUST_PCT}) = {S_RAW:.2f} (int16 units)")

# =========================
# 3) Frame: frame_t is list of instants of video. Two indices of frame_t are separated in time by dt
# =========================
# frames en u (temps vidéo)
dt_u = speed * (1.0 / fps)     # "avance en temps vidéo" par frame
frame_u = np.arange(0.0, U_TOTAL, dt_u)
frame_u = frame_u[frame_u <= U_TOTAL]

# =========================
# 4) Figure: RAW (signal raw, haut) + RAS (raster, bas)
# =========================
fig = plt.figure(figsize=(10, 6), dpi=300)
gs = fig.add_gridspec(2, 1, height_ratios=[1, 1.5], hspace=0.03)
ax_raw = fig.add_subplot(gs[0])
ax_ras = fig.add_subplot(gs[1], sharex=ax_raw)
fig.patch.set_facecolor(BG)
ax_raw.set_facecolor(BG)
ax_ras.set_facecolor(BG)

def setup_axes():
    for ax in (ax_raw, ax_ras):
        ax.set_xlim(-window, +window)
        ax.get_xaxis().set_visible(False)
        ax.get_yaxis().set_visible(False)
        ax.spines[['right', 'top', 'left', 'bottom']].set_visible(False)
        ax.set_facecolor(BG)
    ax_ras.set_ylim(-0.5, n_units - 0.5)
    ax_ras.grid(True, axis="x", alpha=GRID_ALPHA, color=FG)

setup_axes()

# ax_raw.set_ylim(-1.5, (len(raw_ch)-1)*RAW_OFFSET_MULT + 1.5) # Fix Y limits (important sinon autoscale coupe les canaux)
ax_raw.set_ylim(
    -RAW_CLIP - Y_MARGIN,
    (len(raw_ch) - 1) * RAW_OFFSET_MULT + RAW_CLIP + Y_MARGIN)
# Patch stimulation (1 seul par axe, mis à jour à chaque frame)
stim_span_raw = None
stim_span_ras = None

# --- RAW: 4 "lignes" recolorables via LineCollection (1 par canal) ---
raw_collections = []
for ci in range(len(raw_ch)):
    lc = LineCollection([], linewidths=RAW_LINE_W)
    lc.set_alpha(0.95)
    ax_raw.add_collection(lc)
    raw_collections.append(lc)

# (optionnel) on évite cla() et on garde un texte mis à jour
time_text = ax_raw.text(0.01, 1.2, "", transform=ax_raw.transAxes, va="top", color=FG)
stim_text = ax_raw.text(0.01, 1.1, f"Stimulation = {elec} {plots}, {freq}, {intensity}, {round(stim_dur)} s", transform=ax_raw.transAxes, va="top", color=FG)
ax_raw.axvline(0.0, linestyle="--", linewidth=1.0, alpha=0.35, color=FG)


# =========================
# 5) Update
# =========================
def update(k):
    u = float(frame_u[k])
    u0 = u - window
    u1 = u + window

    # ---- RAW ----
    ax_raw.set_xlim(-window, +window)
    ax_raw.set_facecolor(BG)
    # (si besoin, assure une fois au début; sinon tu peux laisser)
    ax_raw.get_xaxis().set_visible(False)
    ax_raw.get_yaxis().set_visible(False)
    ax_raw.spines[['right', 'top', 'left', 'bottom']].set_visible(False)
    # Légende : temps, infos stim :
    # t_real = float(unwarp(np.array([u]))[0])
    t_real = float(u_to_real(np.array([u]))[0])
    time_text.set_text(f"t = {t_real:0.1f} s") # mise à jour du time_text

    # Sélection indices signal sous-échantillonné dans la nouvelle fenêtre temporelle
    i0 = np.searchsorted(u_ds, u0, side="left")
    i1 = np.searchsorted(u_ds, u1, side="right")
    x_rel = u_ds[i0:i1] - u

    segmin = (y_min[i0:i1, :] / S_RAW) * RAW_VISUAL_GAIN
    segmax = (y_max[i0:i1, :] / S_RAW) * RAW_VISUAL_GAIN

    segmin = np.clip(segmin, -RAW_CLIP, RAW_CLIP) # limite les valeurs à RAWCLIP
    segmax = np.clip(segmax, -RAW_CLIP, RAW_CLIP)


    npts = segmin.shape[0]

    offset = RAW_OFFSET_MULT

    # Prépare une recoloration par segment:
    # on trace en "zigzag" min/max -> on a 2*npts points, donc (2*npts - 1) segments
    if npts >= 2:
        # base color = blanc (ou FG)
        base_rgba = np.array(plt.matplotlib.colors.to_rgba(RAW_COLOR), dtype=np.float32)
        base_rgba[3] = RAW_ALPHA

        # empilement vertical des 4 canaux
        # Pour chaque canal, on construit points/segments une fois, puis on colore certains segments
        for ci in range(segmin.shape[1]):
            y = np.empty(npts * 2, dtype=np.float32)
            y[0::2] = segmin[:, ci] + ci * offset
            y[1::2] = segmax[:, ci] + ci * offset
            x = np.repeat(x_rel, 2)

            points = np.column_stack([x, y]).astype(np.float32)                 # (2*npts, 2)
            segments = np.stack([points[:-1], points[1:]], axis=1)              # (2*npts-1, 2, 2)

            colors = np.tile(base_rgba, (segments.shape[0], 1))                 # (nseg, 4)

            # --- recoloration "dans" la courbe autour des spikes ---
            # largeur en points DS autour du spike
            H = HILITE_HALF_PTS

            for ui, st in enumerate(spike_times):
                if st.size == 0:
                    continue
                a = np.searchsorted(spike_u[ui], u - window, side="left")
                b = np.searchsorted(spike_u[ui], u, side="right")

                if b <= a:
                    continue

                spike_rgba = np.array(plt.matplotlib.colors.to_rgba(unit_colors[ui]), dtype=np.float32)
                spike_rgba[3] = 1.0

                # indices locaux j (sur t_ds[i0:i1])
                # on projette les spikes en indices dans t_ds puis on recadre à [0, npts)
                jg = np.searchsorted(u_ds, spike_u[ui][a:b], side="left")
                j = jg - i0
                j = j[(j >= 0) & (j < npts)]
                if j.size == 0:
                    continue

                for jj in j:
                    j0 = max(0, int(jj) - H)
                    j1 = min(npts, int(jj) + H + 1)
                    if j1 - j0 < 2:
                        continue

                    # Mapping points->segments:
                    # points zigzag: index_p = 2*k ou 2*k+1
                    # segments index between consecutive points => segment indices couvrent [2*j0 .. 2*j1-2]
                    s0 = 2 * j0
                    s1 = 2 * j1 - 2
                    s0 = max(0, s0)
                    s1 = min(colors.shape[0] - 1, s1)
                    colors[s0:s1+1, :] = spike_rgba

            # applique à la collection de chaque canal
            raw_collections[ci].set_segments(segments)
            raw_collections[ci].set_color(colors)

    else:
        # fenêtre trop petite => vide
        for lc in raw_collections:
            lc.set_segments([])


    # ---- RASTER ----
    # Clear window + style
    ax_ras.cla()
    ax_ras.set_xlim(-window, +window)
    ax_ras.set_ylim(-0.5, n_units - 0.5)
    ax_ras.get_xaxis().set_visible(False)
    ax_ras.get_yaxis().set_visible(False)
    ax_ras.spines[['right', 'top', 'left', 'bottom']].set_visible(False)
    ax_ras.set_facecolor(BG)
    ax_ras.grid(True, axis="x", alpha=GRID_ALPHA, color=FG)

    # masque la moitié droite (future) : rectangle BG sur [0, +window]
    ax_ras.axvspan(0.0, window, color=BG, alpha=1.0, zorder=10)


    # pour chaque unit, on affiche tous les spikes de la fenetre [l0, t1]
    lo_ras = u - window
    hi_ras = u

    for i, su in enumerate(spike_u):
        if su.size == 0:
            continue
        a = np.searchsorted(su, lo_ras, side="left")
        b = np.searchsorted(su, hi_ras, side="right")
        if b > a:
            x = su[a:b] - u
            ax_ras.vlines(x, i - HALF_H, i + HALF_H, linewidth=LINE_W, color=unit_colors[i], alpha=0.95, zorder=5)

    ax_ras.axvline(0.0, linestyle="--", linewidth=1.0, alpha=0.35, color=FG, zorder=20)

    # ---- PATCH STIMULATION ----
    # d'abord retrait des patchs précédents (sinon accumulation)
    global stim_span_raw, stim_span_ras

    if stim_span_raw is not None:
        try:
            stim_span_raw.remove()
        except Exception: 
            pass
        stim_span_raw = None

    if stim_span_ras is not None:
        try:
            stim_span_ras.remove()
        except Exception:
            pass
        stim_span_ras = None

    # puis on ajoute nouveaux patchs
    xS0 = stim_u0 - u 
    xS1 = stim_u1 - u  
    xs0 = max(-window, xS0)
    xs1 = min(+window, xS1)
    if xs1 > xs0: 
        stim_span_raw = ax_raw.axvspan(xs0, xs1, color="#ff7866", alpha=1, zorder=20)
        stim_span_ras = ax_ras.axvspan(xs0, xs1, color="#ff7866", alpha=1, zorder=20)

    # if u == PRE_SEC :
    #     stim_text.set_color("#ff7866") # affichage label stim en rouge, pendant stim
    # if u == PRE_SEC + STIM_DISPLAY:
    #     stim_text.set_color(FG) # affichage label stim en blanc, après stim
    if pre_t1 <= u < pre_t1 + STIM_DISPLAY:
        stim_text.set_color("#ff7866")   # pendant stim
    else:
        stim_text.set_color(FG)          # avant + après

    return []

anim = FuncAnimation(fig, update, frames=len(frame_u), interval=1000 / fps, blit=False)

# =========================
# 6) Export vidéo + audio mux
# =========================
out_mp4 = Path(f"{root_data}Spike-sorting/Data_folders/{patient}/raw+raster_{patient}_stim{sess}_tt{int(round(tetrode_id))}_{t_start:.0f}-{t_end:.0f}s_middle_stim{STIM_INDEX}.mp4")
tmp_video = out_mp4.with_name(out_mp4.stem + "_silent.mp4")
tmp_wav = out_mp4.with_suffix(".wav")

writer = FFMpegWriter(fps=fps, bitrate=4000)

print(f"Export vidéo silencieuse -> {tmp_video}")
anim.save(str(tmp_video), writer=writer)
plt.close(fig)

print("Synth audio...")
audio = synth_morse_audio_from_spike_u(
    spike_u=spike_u,
    U_TOTAL=U_TOTAL,
    speed=speed,
    sr=SR,
    click_dur=CLICK_DUR,
    base_freq=BASE_FREQ,
    freq_step=FREQ_STEP,
)

write_wav(tmp_wav, audio, sr=SR)

print("Mux audio+vidéo...")
mux_audio_video_ffmpeg(tmp_video, tmp_wav, out_mp4)
print(f"✅ MP4 final -> {out_mp4}")

# Nettoyage
try:
    tmp_video.unlink()
    tmp_wav.unlink()
except Exception:
    pass


                                0         1        2           3
0      TB1-TB21.2mA50.0Hz1025µsec   433.455  5.00198  R Temporal
1      TB4-TB51.2mA50.0Hz1025µsec   805.853  5.00201  R Temporal
2        B1-B21.0mA50.0Hz1025µsec   942.956  5.00201  R Temporal
3      TP1-TP21.2mA50.0Hz1025µsec  1301.160  4.97525  R Temporal
4        B3-B41.2mA50.0Hz1025µsec  1574.880  4.99307  R Temporal
5    OTp8-OTp91.4mA50.0Hz1025µsec  1985.020  4.95743  R Temporal
6  OTp11-OTp121.4mA50.0Hz1025µsec  2059.000  4.97525  R Temporal
7    GPp4-GPp51.4mA50.0Hz1025µsec  2307.690  5.00198  L Temporal
8    GPp5-GPp61.4mA50.0Hz1025µsec  2401.760  4.98416  L Temporal
9    GPp6-GPp71.4mA50.0Hz1025µsec  2724.980  4.99310  L Temporal
Stim choisie: idx=2 label=B1-B21.0mA50.0Hz1025µsec t0=942.956s dur=5.002s
NWB path: /media/aube/Aube2/Spike-sorting/Data_folders/P127_DS76/P127_DS76_stim5/P127_DS76_stim5.nwb
Tétrode 10.0 | units: 7 | keys(ex): [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64

ffmpeg version 7.1.3 Copyright (c) 2000-2025 the FFmpeg developers
  built with gcc 15.2.0 (GCC)
  configuration: --prefix=/usr --libdir=/usr/lib/x86_64-linux-gnu --optflags='-O2 -pipe -g -Wp,-D_FORTIFY_SOURCE=3 -Wp,-D_GLIBCXX_ASSERTIONS -fexceptions -fstack-protector-strong -grecord-gcc-switches -fasynchronous-unwind-tables -fstack-clash-protection -fcf-protection -fno-omit-frame-pointer -mno-omit-leaf-frame-pointer ' --extra-ldflags='-Wl,-z,relro,-z,now -Wl,--as-needed ' --disable-stripping --disable-doc --disable-static --disable-encoders --disable-decoders --enable-shared --enable-gnutls --enable-gcrypt --enable-ladspa --enable-lcms2 --enable-libaom --enable-libdav1d --enable-libmp3lame --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libjxl --enable-libopus --enable-libpulse --enable-libspeex --enable-libtheora --enable-libvorbis --enable-libvpx --enable-librsvg --enable-libwebp --enable-libxml2 --enable-libopenjpeg --enable-libsvtav1 --enable-openal --enab

✅ MP4 final -> /media/aube/Aube2/Spike-sorting/Data_folders/P127_DS76/raw+raster_P127_DS76_stim5_tt10_939-951s_middle_stim2.mp4


[out#0/mp4 @ 0x56304a7f6dc0] video:15014KiB audio:119KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.210247%
frame=  930 fps=0.0 q=-1.0 Lsize=   15165KiB time=00:00:30.93 bitrate=4016.0kbits/s speed=80.3x    
[aac @ 0x56304a68cb40] Qavg: 55823.449


In [ ]:
# une seule tetrode + raster + EEG brut + bande-son + t_spk réinjectés dans EEG. Curseur et apparition au centre
# adapté aux stims.
# export dans le dossier patient

import os
import wave
import subprocess
from pathlib import Path
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.collections import LineCollection
from matplotlib.patches import Rectangle
import pynapple as nap
from pynapple.io.interface_nwb import NWBFile


# =========================
# Paramètres
# =========================
patient = "P127_DS76" #"P111_SL64"
sess = "5"
tetrode_id = 10.0               # 1.0 ... 14.0 (float)
root_data = "/media/aube/Aube2/" # "/home/aube/Documents/article_neuronal_stimic/raster_video/"

# Stimulation
PRE_SEC = 3.0          # secondes avant la stim (temps réel)
POST_SEC = 3.0         # secondes après la stim (temps réel)
STIM_DISPLAY = 0.3     # durée affichée de la stimulation (en secondes vidéo)
STIM_INDEX = 2 # 7         # quelle stimulation dans le fichier (0 = première)

# Vidéo
window = 1 # secondes (x-axis: -window => + window)
speed = 0.2 # 0.3 s pour window de 4 s 
fps = 30

# Raster style
LINE_W = 1.6
HALF_H = 0.45

# Brut .dat
FS = 32768.0
DTYPE = np.int16

# Affichage brut
DARK = True
BG = "#0b0f14"  if DARK else "white" # noir bleuté (moins dur que #000)
FG = "#c7d0d9" if DARK else "black" # texte/grille doux
RAW_COLOR = "#545f69" #FG          # même couleur pour tous les canaux
RAW_ALPHA = 0.95
GRID_ALPHA = 0.10
RAW_LINE_W = 0.2
RAW_DOWNSAMPLE_TARGET = 10000   # points max affichés par frame (perfs)

# autoscale du signal
USE_ROBUST_AUTOSCALE = True
ROBUST_PCT = 99.0       # percentile pour échelle
RAW_OFFSET_MULT = 2.5    # offset vertical entre canaux (en unités int16)

# Audio
SR = 44100
CLICK_DUR = 0.020
BASE_FREQ = 450.0
FREQ_STEP = 70.0

# =========================
# Load NWB et stims
# =========================
def get_nwb(patient, session, root=""): # root="/media/aube/Aube/"
    path_folder = root + "Spike-sorting/Data_folders/" + patient + "/" + patient + "_stim" + session + "/"
    files_basename = patient + "_stim" + session
    nwbfile_path = path_folder + files_basename + ".nwb"
    print("NWB path:", nwbfile_path)

    if not os.path.exists(nwbfile_path):
        from datetime import datetime
        from dateutil import tz
        from neuroconv.datainterfaces import NeuroScopeSortingInterface

        xml_path = Path(path_folder + files_basename + ".xml")
        interface = NeuroScopeSortingInterface(folder_path=Path(path_folder), xml_file_path=xml_path, verbose=False)
        metadata = interface.get_metadata()
        session_start_time = datetime(2023, 4, 4, 12, 30, 0, tzinfo=tz.gettz("US/Pacific"))
        metadata["NWBFile"].update(session_start_time=session_start_time)
        interface.run_conversion(nwbfile_path=nwbfile_path, metadata=metadata)

    return NWBFile(nwbfile_path)["units"], Path(nwbfile_path)

def get_stim(patient, session, root="/home/aube/Documents/article_neuronal_stimic/raster_video/"): # root="/media/aube/Aube/"
    stim_path = root + "Spike-sorting/Data_folders/" + patient + "/" + patient + "_stim" + session + "/" + f"{patient}_stim{sess}_stim_events_TRC_re-shifted_loca.txt"
    if not os.path.exists(stim_path):
        raise FileNotFoundError(f"Fichier stim introuvable: {stim_path}")

    # adapte le sep si nécessaire (tab/espaces)
    stim_df = pd.read_csv(stim_path, sep=", ", engine="python", header=None)
    print(stim_df)
    stim_df.columns = ["label", "t_start", "dur", "loca"]

    # sélection
    if STIM_INDEX < 0 or STIM_INDEX >= len(stim_df):
        raise ValueError(f"STIM_INDEX invalide ({STIM_INDEX}) — nb stims={len(stim_df)}")

    stim_label = stim_df.loc[STIM_INDEX, "label"]
    stim_t0 = float(stim_df.loc[STIM_INDEX, "t_start"])
    stim_dur = float(stim_df.loc[STIM_INDEX, "dur"])
    stim_t1 = stim_t0 + stim_dur

    print(f"Stim choisie: idx={STIM_INDEX} label={stim_label} t0={stim_t0:.3f}s dur={stim_dur:.3f}s")
    return [stim_label, stim_t0, stim_dur, stim_t1]

# =========================
# Outils brut .dat (memmap)
# =========================
def open_dat_memmap(dat_path: Path, n_channels: int, dtype=np.int16):
    # Attention : marche que si tous les canaux sont inclus dans le .dat
    if not dat_path.exists():
        raise FileNotFoundError(f".dat introuvable: {dat_path.resolve()}")

    bytes_per_val = np.dtype(dtype).itemsize
    n_vals = dat_path.stat().st_size // bytes_per_val
    if n_vals % n_channels != 0:
        raise ValueError(
            f"Taille .dat incompatible: n_vals={n_vals} n'est pas multiple de n_channels={n_channels}."
        )
    n_samples = n_vals // n_channels
    mm = np.memmap(dat_path, dtype=dtype, mode="r", shape=(n_samples, n_channels), order="C")
    return mm, n_samples

def tetrode_channels(tetrode_id):
    # tetrode_id peut être float -> cast int
    tt = int(round(float(tetrode_id)))
    if tt < 1 or tt > N_TETRODES:
        raise ValueError(f"tetrode_id={tetrode_id} invalide. Attendu: 1..{N_TETRODES}.")
    start = (tt - 1) * 4
    return list(range(start, start + 4))

def get_raw_window(mm, fs, t0, t1, channels, t_min_valid=0.0, target_points=2500):
    """
    Fenêtre [t0, t1] en temps ABSOLU, avec padding NaN si t0 < t_min_valid.
    Retourne x_rel in [-window, 0], seg shape (n_points, n_ch) en float32.
    """
    n_samples_total = mm.shape[0]
    n_ch = len(channels)

    # indices en samples (peuvent être <0)
    s0 = int(np.floor(t0 * fs))
    s1 = int(np.floor(t1 * fs))

    s_min = int(np.floor(t_min_valid * fs))

    # longueur cible en samples (fenêtre constante)
    n_target = max(1, int(np.round((t1 - t0) * fs)))

    # padding à gauche si on est avant t_min_valid
    left_pad = max(0, s_min - s0)

    # clamp lecture
    s0_read = max(s_min, s0)
    s1_read = min(n_samples_total, s1)

    if s1_read > s0_read:
        seg = mm[s0_read:s1_read, channels].astype(np.float32)
    else:
        seg = np.empty((0, n_ch), dtype=np.float32)

    # ajoute padding NaN à gauche
    if left_pad > 0:
        seg = np.vstack([np.full((left_pad, n_ch), np.nan, dtype=np.float32), seg])

    # ajuste à n_target (pad NaN à droite si besoin)
    if seg.shape[0] > n_target:
        seg = seg[:n_target]
    elif seg.shape[0] < n_target:
        pad = n_target - seg.shape[0]
        seg = np.vstack([seg, np.full((pad, n_ch), np.nan, dtype=np.float32)])

    # downsample pour affichage
    n = seg.shape[0]
    if n > target_points:
        step = int(np.ceil(n / target_points))
        seg = seg[::step, :]
        # x_rel recalculé avec step
        x_rel = (np.arange(seg.shape[0], dtype=np.float32) * (step / fs)) - (t1 - t0)
    else:
        x_rel = (np.arange(n, dtype=np.float32) / fs) - (t1 - t0)

    return x_rel, seg

# =========================
# Audio (morse)
# =========================
def synth_morse_audio_from_spike_u(spike_u, U_TOTAL, speed, sr=44100,
                                   click_dur=0.020, base_freq=450.0, freq_step=70.0):
    duration_video = float(U_TOTAL) / float(speed)
    n = int(np.ceil(duration_video * sr))
    audio = np.zeros(n, dtype=np.float32)

    L = max(1, int(click_dur * sr))
    tt = np.arange(L, dtype=np.float32) / sr

    env = np.exp(-tt * 45.0).astype(np.float32)
    attack = np.minimum(1.0, tt / 0.002).astype(np.float32)
    env *= attack

    for i, su in enumerate(spike_u):
        if su.size == 0:
            continue
        f = base_freq + i * freq_step
        tone = (np.sin(2 * np.pi * f * tt) * env).astype(np.float32)

        for u_spk in su:
            # u_spk est en [0, U_TOTAL]
            idx0 = int((float(u_spk) / speed) * sr)
            if idx0 < 0 or idx0 >= n:
                continue
            idx1 = min(n, idx0 + L)
            audio[idx0:idx1] += tone[: idx1 - idx0]

    peak = float(np.max(np.abs(audio))) if audio.size else 0.0
    if peak > 0:
        audio /= (peak * 1.05)
    return audio


def write_wav(path, audio, sr=44100):
    audio_i16 = np.int16(np.clip(audio, -1.0, 1.0) * 32767)
    with wave.open(str(path), "wb") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(sr)
        wf.writeframes(audio_i16.tobytes())

def mux_audio_video_ffmpeg(video_in, audio_in, video_out):
    cmd = [
        "ffmpeg", "-y",
        "-i", str(video_in),
        "-i", str(audio_in),
        "-c:v", "copy",
        "-c:a", "aac",
        "-shortest",
        str(video_out),
    ]
    subprocess.run(cmd, check=True)


# =========================
# 1) Charger stim, et fonctions de warp (réel ↔ vidéo)
# =========================
stim_carac = get_stim(patient, sess, root_data)
stim_label, stim_t0, stim_dur, stim_t1 = stim_carac[0], stim_carac[1], stim_carac[2], stim_carac[3]
t_start = stim_t0 - PRE_SEC
t_end   = stim_t1 + POST_SEC
elec_plot = stim_label[:stim_label.index('mA')-3]
elec = re.match(r"([A-Za-z'_]+)(\d+)-([A-Za-z'_]+)(\d+)", elec_plot).group(1).replace('_','')
plots = re.match(r"([A-Za-z'_]+)(\d+)-([A-Za-z'_]+)(\d+)", elec_plot).group(2) +'-'+ re.match(r"([A-Za-z'_]+)(\d+)-([A-Za-z'_]+)(\d+)", elec_plot).group(4)
freq = stim_label[stim_label.index('mA')+2:stim_label.index('Hz')-2]+' '+stim_label[stim_label.index('Hz'):stim_label.index('Hz')+2]
intensity = stim_label[stim_label.index('mA')-3:stim_label.index('mA')]+' '+stim_label[stim_label.index('mA'):stim_label.index('mA')+2]

# temps "vidéo" u=0 au début de la fenêtre réelle
r0 = t_start
r_pre_end = stim_t0
r_stim_end = stim_t1
r1 = t_end

U_PRE = PRE_SEC
U_STIM = STIM_DISPLAY
U_POST = POST_SEC
U_TOTAL = U_PRE + U_STIM + U_POST

stim_u0 = U_PRE
stim_u1 = U_PRE + U_STIM

def warp(t):
    """t (sec réel) -> u (sec vidéo), piecewise linéaire."""
    t = np.asarray(t, dtype=np.float64)
    u = np.empty_like(t)

    # pre: slope 1
    m1 = (t <= r_pre_end)
    u[m1] = (t[m1] - r0)

    # stim: compressed
    m2 = (t > r_pre_end) & (t <= r_stim_end)
    if stim_dur <= 0:
        raise ValueError("durée stim <= 0 dans le fichier")
    u[m2] = U_PRE + (t[m2] - r_pre_end) * (U_STIM / stim_dur)

    # post: slope 1
    m3 = (t > r_stim_end)
    u[m3] = (U_PRE + U_STIM) + (t[m3] - r_stim_end)

    return u

def unwarp(u):
    """u (sec vidéo) -> t (sec réel). (si tu en as besoin)"""
    u = np.asarray(u, dtype=np.float64)
    t = np.empty_like(u)

    m1 = (u <= U_PRE)
    t[m1] = r0 + u[m1]

    m2 = (u > U_PRE) & (u <= U_PRE + U_STIM)
    t[m2] = r_pre_end + (u[m2] - U_PRE) * (stim_dur / U_STIM)

    m3 = (u > U_PRE + U_STIM)
    t[m3] = r_stim_end + (u[m3] - (U_PRE + U_STIM))

    return t


# =========================
# 1) Charger spikes, filtrer tétrode
# =========================
spk, nwb_path = get_nwb(patient, sess, root_data)

epoch_scroll = nap.IntervalSet(start=[t_start], end=[t_end], time_units="s")
spk = spk.restrict(epoch_scroll)

groups_dict = spk.getby_category("group")
available = sorted([float(g) for g in groups_dict.keys()])
if float(tetrode_id) not in [float(g) for g in groups_dict.keys()]:
    raise ValueError(
        f"Aucune unit pour tetrode/group={tetrode_id}. Groupes dispo (intervalle): {available}"
    )

idx_keep = np.asarray(groups_dict[float(tetrode_id)].index, dtype=int)
spk = spk[idx_keep]
unit_keys = list(spk.index)

spike_times = []
for k in unit_keys:
    spike_times.append(np.sort(np.asarray(spk[k].t, dtype=float)))

n_units = len(spike_times)
print(f"Tétrode {tetrode_id} | units: {n_units} | keys(ex): {unit_keys[:8]}")

# Couleurs par unit
if n_units < 12:
    cmap_units = plt.get_cmap("Set3")  # très lisible sur fond sombre #plt.get_cmap("tab20")
    unit_colors = [cmap_units(i % cmap_units.N) for i in range(n_units)]
else:
    import matplotlib.colors as mcolors
    def lighten(color, amount=0.55): # on éclaircit une colormap plus foncée (0.55 trop éclairci)
        r, g, b, a = mcolors.to_rgba(color)
        r = r + (1 - r) * amount
        g = g + (1 - g) * amount
        b = b + (1 - b) * amount
        return (r, g, b, a)

    cmap_units = plt.get_cmap("tab20")
    unit_colors = [lighten(cmap_units(i % cmap_units.N), amount=0.2) for i in range(n_units)]


# =========================
# 2) Ouvrir brut .dat
# =========================
dat_path = nwb_path.with_suffix(".dat")
N_TETRODES = pd.read_csv(root_data + "Spike-sorting/Data_folders/" + patient + "/mapping_anat_" + patient + '.txt').shape[0]
N_CHANNELS = N_TETRODES * 4 
mm, n_samples = open_dat_memmap(dat_path, n_channels=N_CHANNELS, dtype=DTYPE) # matrice dat,  nb de samples d'un canal
dur_dat = n_samples / FS
print(f"DAT: {dat_path.name} | n_samples={n_samples} | durée~{dur_dat:.2f}s | n_channels={N_CHANNELS}")

# Vérif temps
t_start = max(0.0, float(t_start))
t_end = min(float(t_end), float(dur_dat))
if t_end <= t_start:
    raise ValueError(f"t_end ({t_end}) doit être > t_start ({t_start}).")

# Canaux de la tétrode
raw_ch = tetrode_channels(tetrode_id)
print(f"Canaux bruts tétrode {tetrode_id}: {raw_ch}")

def precompute_minmax_envelope(mm, fs, t_start, t_end, channels, ds_fs=1000):
    """
    Produit une enveloppe min/max à fréquence ds_fs (Hz) sur [t_start, t_end].
    Retour:
      t_ds (sec absolues), ymin (n_pts, n_ch), ymax (n_pts, n_ch)
    """
    s0 = int(np.floor(t_start * fs))
    s1 = int(np.floor(t_end * fs))
    s0 = max(0, s0); s1 = min(mm.shape[0], s1)
    if s1 <= s0:
        raise ValueError("Intervalle brut vide (vérifie t_start/t_end).")

    # taille de bloc en samples
    block = max(1, int(round(fs / ds_fs)))
    n = s1 - s0
    n_blocks = n // block

    # on tronque à un multiple de block pour reshape
    n_use = n_blocks * block
    X = mm[s0:s0+n_use, channels].astype(np.float32)  # (n_use, n_ch)
    X = X.reshape(n_blocks, block, len(channels))

    ymin = np.min(X, axis=1)
    ymax = np.max(X, axis=1)
    # temps au centre de chaque bloc
    t_ds = t_start + (np.arange(n_blocks, dtype=np.float32) + 0.5) * (block / fs)
    return t_ds, ymin, ymax, block

# ex: downsampling a 1000 Hz d'affichage (ok aussi pour 500 ou 2000)
DS_FS = 2000
HILITE_MS = 6
HILITE_HALF_PTS = max(1, int(round(HILITE_MS * DS_FS / 1000.0)))

t_ds, y_min, y_max, block = precompute_minmax_envelope(mm, FS, t_start, t_end, raw_ch, ds_fs=DS_FS)
print(f"RAW downsample précomputé: {DS_FS} Hz | points={len(t_ds)} | block={block} samples")

# Conversion du signal et des spikes en temps warpé :
u_ds = warp(t_ds)  # même longueur que t_ds, monotone
spike_u = [warp(st) for st in spike_times]  # liste d'arrays en temps vidéo


# ---- gain/scale RAW fixe sur tout l'intervalle (stable) basé sur signal pré-stim ----
def estimate_global_scale(mm, fs, t_start, t_end, channels, n_probe=80_000, pct=99.0):
    # on prend quelques samples répartis uniformément sur l'intervalle
    s0 = int(t_start * fs)
    s1 = int(t_end * fs)
    s0 = max(0, s0); s1 = min(mm.shape[0], s1)
    if s1 <= s0:
        return 1.0
    n_total = s1 - s0
    step = max(1, n_total // n_probe)
    seg = mm[s0:s1:step, channels].astype(np.float32)
    val = np.nanpercentile(np.abs(seg), pct)
    if not np.isfinite(val) or val <= 0:
        val = 1.0
    return float(val)

S_RAW = estimate_global_scale(mm, FS, t_start, t_start+PRE_SEC, raw_ch, n_probe=120_000, pct=ROBUST_PCT)
print(f"RAW scale (p{ROBUST_PCT}) = {S_RAW:.2f} (int16 units)")

# =========================
# 3) Frame: frame_t is list of instants of video. Two indices of frame_t are separated in time by dt
# =========================
# frames en u (temps vidéo)
dt_u = speed * (1.0 / fps)     # "avance en temps vidéo" par frame
frame_u = np.arange(0.0, U_TOTAL, dt_u)
frame_u = frame_u[frame_u <= U_TOTAL]

# =========================
# 4) Figure: RAW (signal raw, haut) + RAS (raster, bas)
# =========================
fig = plt.figure(figsize=(10, 6), dpi=300)
gs = fig.add_gridspec(2, 1, height_ratios=[1, 1.5], hspace=0.03)
ax_raw = fig.add_subplot(gs[0])
ax_ras = fig.add_subplot(gs[1], sharex=ax_raw)
fig.patch.set_facecolor(BG)
ax_raw.set_facecolor(BG)
ax_ras.set_facecolor(BG)

def setup_axes():
    for ax in (ax_raw, ax_ras):
        ax.set_xlim(-window, +window)
        ax.get_xaxis().set_visible(False)
        ax.get_yaxis().set_visible(False)
        ax.spines[['right', 'top', 'left', 'bottom']].set_visible(False)
        ax.set_facecolor(BG)
    ax_ras.set_ylim(-0.5, n_units - 0.5)
    ax_ras.grid(True, axis="x", alpha=GRID_ALPHA, color=FG)

setup_axes()

ax_raw.set_ylim(-1.5, (len(raw_ch)-1)*RAW_OFFSET_MULT + 1.5) # Fix Y limits (important sinon autoscale coupe les canaux)

# Patch stimulation (1 seul par axe, mis à jour à chaque frame)
stim_span_raw = None
stim_span_ras = None

# --- RAW: 4 "lignes" recolorables via LineCollection (1 par canal) ---
raw_collections = []
for ci in range(len(raw_ch)):
    lc = LineCollection([], linewidths=RAW_LINE_W)
    lc.set_alpha(0.95)
    ax_raw.add_collection(lc)
    raw_collections.append(lc)

# (optionnel) on évite cla() et on garde un texte mis à jour
time_text = ax_raw.text(0.01, 1.2, "", transform=ax_raw.transAxes, va="top", color=FG)
stim_text = ax_raw.text(0.01, 1.1, f"Stimulation = {elec} {plots}, {freq}, {intensity}, {round(stim_dur)} s", transform=ax_raw.transAxes, va="top", color=FG)
ax_raw.axvline(0.0, linestyle="--", linewidth=1.0, alpha=0.35, color=FG)


# =========================
# 5) Update
# =========================
def update(k):
    u = float(frame_u[k])
    u0 = u - window
    u1 = u + window

    # ---- RAW ----
    ax_raw.set_xlim(-window, +window)
    ax_raw.set_facecolor(BG)
    # (si besoin, assure une fois au début; sinon tu peux laisser)
    ax_raw.get_xaxis().set_visible(False)
    ax_raw.get_yaxis().set_visible(False)
    ax_raw.spines[['right', 'top', 'left', 'bottom']].set_visible(False)
    # Légende : temps, infos stim :
    t_real = float(unwarp(np.array([u]))[0])
    time_text.set_text(f"t = {t_real:0.1f} s") # mise à jour du time_text

    # Sélection indices signal sous-échantillonné dans la nouvelle fenêtre temporelle
    i0 = np.searchsorted(u_ds, u0, side="left")
    i1 = np.searchsorted(u_ds, u1, side="right")
    x_rel = u_ds[i0:i1] - u

    segmin = y_min[i0:i1, :] / S_RAW
    segmax = y_max[i0:i1, :] / S_RAW

    npts = segmin.shape[0]

    offset = RAW_OFFSET_MULT

    # Prépare une recoloration par segment:
    # on trace en "zigzag" min/max -> on a 2*npts points, donc (2*npts - 1) segments
    if npts >= 2:
        # base color = blanc (ou FG)
        base_rgba = np.array(plt.matplotlib.colors.to_rgba(RAW_COLOR), dtype=np.float32)
        base_rgba[3] = RAW_ALPHA

        # empilement vertical des 4 canaux
        # Pour chaque canal, on construit points/segments une fois, puis on colore certains segments
        for ci in range(segmin.shape[1]):
            y = np.empty(npts * 2, dtype=np.float32)
            y[0::2] = segmin[:, ci] + ci * offset
            y[1::2] = segmax[:, ci] + ci * offset
            x = np.repeat(x_rel, 2)

            points = np.column_stack([x, y]).astype(np.float32)                 # (2*npts, 2)
            segments = np.stack([points[:-1], points[1:]], axis=1)              # (2*npts-1, 2, 2)

            colors = np.tile(base_rgba, (segments.shape[0], 1))                 # (nseg, 4)

            # --- recoloration "dans" la courbe autour des spikes ---
            # largeur en points DS autour du spike
            H = HILITE_HALF_PTS

            for ui, st in enumerate(spike_times):
                if st.size == 0:
                    continue
                a = np.searchsorted(spike_u[ui], u - window, side="left")
                b = np.searchsorted(spike_u[ui], u, side="right")

                if b <= a:
                    continue

                spike_rgba = np.array(plt.matplotlib.colors.to_rgba(unit_colors[ui]), dtype=np.float32)
                spike_rgba[3] = 1.0

                # indices locaux j (sur t_ds[i0:i1])
                # on projette les spikes en indices dans t_ds puis on recadre à [0, npts)
                jg = np.searchsorted(u_ds, spike_u[ui][a:b], side="left")
                j = jg - i0
                j = j[(j >= 0) & (j < npts)]
                if j.size == 0:
                    continue

                for jj in j:
                    j0 = max(0, int(jj) - H)
                    j1 = min(npts, int(jj) + H + 1)
                    if j1 - j0 < 2:
                        continue

                    # Mapping points->segments:
                    # points zigzag: index_p = 2*k ou 2*k+1
                    # segments index between consecutive points => segment indices couvrent [2*j0 .. 2*j1-2]
                    s0 = 2 * j0
                    s1 = 2 * j1 - 2
                    s0 = max(0, s0)
                    s1 = min(colors.shape[0] - 1, s1)
                    colors[s0:s1+1, :] = spike_rgba

            # applique à la collection de chaque canal
            raw_collections[ci].set_segments(segments)
            raw_collections[ci].set_color(colors)

    else:
        # fenêtre trop petite => vide
        for lc in raw_collections:
            lc.set_segments([])


    # ---- RASTER ----
    # Clear window + style
    ax_ras.cla()
    ax_ras.set_xlim(-window, +window)
    ax_ras.set_ylim(-0.5, n_units - 0.5)
    ax_ras.get_xaxis().set_visible(False)
    ax_ras.get_yaxis().set_visible(False)
    ax_ras.spines[['right', 'top', 'left', 'bottom']].set_visible(False)
    ax_ras.set_facecolor(BG)
    ax_ras.grid(True, axis="x", alpha=GRID_ALPHA, color=FG)

    # masque la moitié droite (future) : rectangle BG sur [0, +window]
    ax_ras.axvspan(0.0, window, color=BG, alpha=1.0, zorder=10)


    # pour chaque unit, on affiche tous les spikes de la fenetre [l0, t1]
    lo_ras = u - window
    hi_ras = u

    for i, su in enumerate(spike_u):
        if su.size == 0:
            continue
        a = np.searchsorted(su, lo_ras, side="left")
        b = np.searchsorted(su, hi_ras, side="right")
        if b > a:
            x = su[a:b] - u
            ax_ras.vlines(x, i - HALF_H, i + HALF_H, linewidth=LINE_W, color=unit_colors[i], alpha=0.95, zorder=5)

    ax_ras.axvline(0.0, linestyle="--", linewidth=1.0, alpha=0.35, color=FG, zorder=20)

    # ---- PATCH STIMULATION ----
    # d'abord retrait des patchs précédents (sinon accumulation)
    global stim_span_raw, stim_span_ras

    if stim_span_raw is not None:
        try:
            stim_span_raw.remove()
        except Exception:
            pass
        stim_span_raw = None

    if stim_span_ras is not None:
        try:
            stim_span_ras.remove()
        except Exception:
            pass
        stim_span_ras = None

    # puis on ajoute nouveaux patchs
    xS0 = stim_u0 - u 
    xS1 = stim_u1 - u  
    xs0 = max(-window, xS0)
    xs1 = min(+window, xS1)
    if xs1 > xs0: 
        stim_span_raw = ax_raw.axvspan(xs0, xs1, color="#ff7866", alpha=0.2, zorder=20)
        stim_span_ras = ax_ras.axvspan(xs0, xs1, color="#ff7866", alpha=0.2, zorder=20)

    # if u == PRE_SEC :
    #     stim_text.set_color("#ff7866") # affichage label stim en rouge, pendant stim
    # if u == PRE_SEC + STIM_DISPLAY:
    #     stim_text.set_color(FG) # affichage label stim en blanc, après stim
    if PRE_SEC <= u < PRE_SEC + STIM_DISPLAY:
        stim_text.set_color("#ff7866")   # pendant stim
    else:
        stim_text.set_color(FG)          # avant + après

    return []

anim = FuncAnimation(fig, update, frames=len(frame_u), interval=1000 / fps, blit=False)

# =========================
# 6) Export vidéo + audio mux
# =========================
out_mp4 = Path(f"{root_data}Spike-sorting/Data_folders/{patient}/raw+raster_{patient}_stim{sess}_tt{int(round(tetrode_id))}_{t_start:.0f}-{t_end:.0f}s_middle_stim{STIM_INDEX}.mp4")
tmp_video = out_mp4.with_name(out_mp4.stem + "_silent.mp4")
tmp_wav = out_mp4.with_suffix(".wav")

writer = FFMpegWriter(fps=fps, bitrate=4000)

print(f"Export vidéo silencieuse -> {tmp_video}")
anim.save(str(tmp_video), writer=writer)
plt.close(fig)

print("Synth audio...")
audio = synth_morse_audio_from_spike_u(
    spike_u=spike_u,
    U_TOTAL=U_TOTAL,
    speed=speed,
    sr=SR,
    click_dur=CLICK_DUR,
    base_freq=BASE_FREQ,
    freq_step=FREQ_STEP,
)

write_wav(tmp_wav, audio, sr=SR)

print("Mux audio+vidéo...")
mux_audio_video_ffmpeg(tmp_video, tmp_wav, out_mp4)
print(f"✅ MP4 final -> {out_mp4}")

# Nettoyage
try:
    tmp_video.unlink()
    tmp_wav.unlink()
except Exception:
    pass


                                0         1        2           3
0      TB1-TB21.2mA50.0Hz1025µsec   433.455  5.00198  R Temporal
1      TB4-TB51.2mA50.0Hz1025µsec   805.853  5.00201  R Temporal
2        B1-B21.0mA50.0Hz1025µsec   942.956  5.00201  R Temporal
3      TP1-TP21.2mA50.0Hz1025µsec  1301.160  4.97525  R Temporal
4        B3-B41.2mA50.0Hz1025µsec  1574.880  4.99307  R Temporal
5    OTp8-OTp91.4mA50.0Hz1025µsec  1985.020  4.95743  R Temporal
6  OTp11-OTp121.4mA50.0Hz1025µsec  2059.000  4.97525  R Temporal
7    GPp4-GPp51.4mA50.0Hz1025µsec  2307.690  5.00198  L Temporal
8    GPp5-GPp61.4mA50.0Hz1025µsec  2401.760  4.98416  L Temporal
9    GPp6-GPp71.4mA50.0Hz1025µsec  2724.980  4.99310  L Temporal
Stim choisie: idx=2 label=B1-B21.0mA50.0Hz1025µsec t0=942.956s dur=5.002s
NWB path: /media/aube/Aube2/Spike-sorting/Data_folders/P127_DS76/P127_DS76_stim5/P127_DS76_stim5.nwb
Tétrode 10.0 | units: 7 | keys(ex): [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64

In [ ]:
# une seule tetrode + raster + EEG brut + bande-son + t_spk réinjectés dans EEG. Curseur et apparition au centre

import os
import wave
import subprocess
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.collections import LineCollection
import pynapple as nap
from pynapple.io.interface_nwb import NWBFile


# =========================
# Paramètres
# =========================
t_start, t_end = 2160, 2175        # secondes (temps absolu)
patient = "P111_SL64"
sess = "5"
tetrode_id = 13.0               # 1..14 (float)

window = 2 # secondes
speed = 0.2 # 0.3 s pour window de 4 s 
fps = 25

# Raster style
LINE_W = 1.6
HALF_H = 0.45

# Brut .dat
FS = 32768.0
DTYPE = np.int16
N_TETRODES = 14
N_CHANNELS = N_TETRODES * 4  # 56

# Affichage brut
DARK = True
BG = "#0b0f14"  if DARK else "white" # noir bleuté (moins dur que #000)
FG = "#c7d0d9" if DARK else "black" # texte/grille doux
RAW_COLOR = "#545f69" #FG          # même couleur pour tous les canaux
RAW_ALPHA = 0.95
GRID_ALPHA = 0.10
RAW_LINE_W = 0.2
RAW_DOWNSAMPLE_TARGET = 10000   # points max affichés par frame (perfs)

# autoscale robuste
USE_ROBUST_AUTOSCALE = True
ROBUST_PCT = 99.0       # percentile pour échelle
RAW_OFFSET_MULT = 2.5    # offset vertical entre canaux (en unités int16)

# Audio
SR = 44100
CLICK_DUR = 0.020
BASE_FREQ = 450.0
FREQ_STEP = 70.0


# =========================
# Load NWB
# =========================
def get_nwb(patient, session, root=""): # root="/media/aube/Aube/"
    path_folder = root + "Spike-sorting/Data_folders/" + patient + "/" + patient + "_stim" + session + "/"
    files_basename = patient + "_stim" + session
    nwbfile_path = path_folder + files_basename + ".nwb"
    print("NWB path:", nwbfile_path)

    if not os.path.exists(nwbfile_path):
        from datetime import datetime
        from dateutil import tz
        from neuroconv.datainterfaces import NeuroScopeSortingInterface

        xml_path = Path(path_folder + files_basename + ".xml")
        interface = NeuroScopeSortingInterface(folder_path=Path(path_folder), xml_file_path=xml_path, verbose=False)
        metadata = interface.get_metadata()
        session_start_time = datetime(2023, 4, 4, 12, 30, 0, tzinfo=tz.gettz("US/Pacific"))
        metadata["NWBFile"].update(session_start_time=session_start_time)
        interface.run_conversion(nwbfile_path=nwbfile_path, metadata=metadata)

    return NWBFile(nwbfile_path)["units"], Path(nwbfile_path)


# =========================
# Outils brut .dat (memmap)
# =========================
def open_dat_memmap(dat_path: Path, n_channels: int, dtype=np.int16):
    if not dat_path.exists():
        raise FileNotFoundError(f".dat introuvable: {dat_path.resolve()}")

    bytes_per_val = np.dtype(dtype).itemsize
    n_vals = dat_path.stat().st_size // bytes_per_val
    if n_vals % n_channels != 0:
        raise ValueError(
            f"Taille .dat incompatible: n_vals={n_vals} n'est pas multiple de n_channels={n_channels}."
        )
    n_samples = n_vals // n_channels
    mm = np.memmap(dat_path, dtype=dtype, mode="r", shape=(n_samples, n_channels), order="C")
    return mm, n_samples

def tetrode_channels(tetrode_id):
    # tetrode_id peut être float -> cast int
    tt = int(round(float(tetrode_id)))
    if tt < 1 or tt > N_TETRODES:
        raise ValueError(f"tetrode_id={tetrode_id} invalide. Attendu: 1..{N_TETRODES}.")
    start = (tt - 1) * 4
    return list(range(start, start + 4))

def get_raw_window(mm, fs, t0, t1, channels, t_min_valid=0.0, target_points=2500):
    """
    Fenêtre [t0, t1] en temps ABSOLU, avec padding NaN si t0 < t_min_valid.
    Retourne x_rel in [-window, 0], seg shape (n_points, n_ch) en float32.
    """
    n_samples_total = mm.shape[0]
    n_ch = len(channels)

    # indices en samples (peuvent être <0)
    s0 = int(np.floor(t0 * fs))
    s1 = int(np.floor(t1 * fs))

    s_min = int(np.floor(t_min_valid * fs))

    # longueur cible en samples (fenêtre constante)
    n_target = max(1, int(np.round((t1 - t0) * fs)))

    # padding à gauche si on est avant t_min_valid
    left_pad = max(0, s_min - s0)

    # clamp lecture
    s0_read = max(s_min, s0)
    s1_read = min(n_samples_total, s1)

    if s1_read > s0_read:
        seg = mm[s0_read:s1_read, channels].astype(np.float32)
    else:
        seg = np.empty((0, n_ch), dtype=np.float32)

    # ajoute padding NaN à gauche
    if left_pad > 0:
        seg = np.vstack([np.full((left_pad, n_ch), np.nan, dtype=np.float32), seg])

    # ajuste à n_target (pad NaN à droite si besoin)
    if seg.shape[0] > n_target:
        seg = seg[:n_target]
    elif seg.shape[0] < n_target:
        pad = n_target - seg.shape[0]
        seg = np.vstack([seg, np.full((pad, n_ch), np.nan, dtype=np.float32)])

    # downsample pour affichage
    n = seg.shape[0]
    if n > target_points:
        step = int(np.ceil(n / target_points))
        seg = seg[::step, :]
        # x_rel recalculé avec step
        x_rel = (np.arange(seg.shape[0], dtype=np.float32) * (step / fs)) - (t1 - t0)
    else:
        x_rel = (np.arange(n, dtype=np.float32) / fs) - (t1 - t0)

    return x_rel, seg

# =========================
# Audio (morse)
# =========================
def synth_morse_audio_from_spikes(spike_times, t_start, t_end, speed, sr=44100,
                                  click_dur=0.020, base_freq=450.0, freq_step=70.0):
    duration_video = float(t_end - t_start) / float(speed)
    n = int(np.ceil(duration_video * sr))
    audio = np.zeros(n, dtype=np.float32)

    L = max(1, int(click_dur * sr))
    tt = np.arange(L, dtype=np.float32) / sr

    env = np.exp(-tt * 45.0).astype(np.float32)
    attack = np.minimum(1.0, tt / 0.002).astype(np.float32)
    env *= attack

    # pour chaque unit, une tonalité
    for i, st in enumerate(spike_times):
        if st.size == 0:
            continue
        f = base_freq + i * freq_step
        tone = (np.sin(2 * np.pi * f * tt) * env).astype(np.float32)

        st_in = st[(st >= t_start) & (st <= t_end)]
        for s in st_in:
            idx0 = int(((float(s) - t_start) / speed) * sr)
            if idx0 < 0 or idx0 >= n:
                continue
            idx1 = min(n, idx0 + L)
            audio[idx0:idx1] += tone[: idx1 - idx0]

    peak = float(np.max(np.abs(audio))) if audio.size else 0.0
    if peak > 0:
        audio /= (peak * 1.05)
    return audio

def write_wav(path, audio, sr=44100):
    audio_i16 = np.int16(np.clip(audio, -1.0, 1.0) * 32767)
    with wave.open(str(path), "wb") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(sr)
        wf.writeframes(audio_i16.tobytes())

def mux_audio_video_ffmpeg(video_in, audio_in, video_out):
    cmd = [
        "ffmpeg", "-y",
        "-i", str(video_in),
        "-i", str(audio_in),
        "-c:v", "copy",
        "-c:a", "aac",
        "-shortest",
        str(video_out),
    ]
    subprocess.run(cmd, check=True)


# =========================
# 1) Charger spikes, filtrer tétrode
# =========================
spk, nwb_path = get_nwb(patient, sess)

epoch_scroll = nap.IntervalSet(start=[t_start], end=[t_end], time_units="s")
spk = spk.restrict(epoch_scroll)

groups_dict = spk.getby_category("group")
available = sorted([float(g) for g in groups_dict.keys()])
if float(tetrode_id) not in [float(g) for g in groups_dict.keys()]:
    raise ValueError(
        f"Aucune unit pour tetrode/group={tetrode_id}. Groupes dispo (intervalle): {available}"
    )

idx_keep = np.asarray(groups_dict[float(tetrode_id)].index, dtype=int)
spk = spk[idx_keep]
unit_keys = list(spk.index)

spike_times = []
for k in unit_keys:
    spike_times.append(np.sort(np.asarray(spk[k].t, dtype=float)))

n_units = len(spike_times)
print(f"Tétrode {tetrode_id} | units: {n_units} | keys(ex): {unit_keys[:8]}")

# Couleurs par unit
if n_units < 12:
    cmap_units = plt.get_cmap("Set3")  # très lisible sur fond sombre #plt.get_cmap("tab20")
    unit_colors = [cmap_units(i % cmap_units.N) for i in range(n_units)]
else:
    import matplotlib.colors as mcolors
    def lighten(color, amount=0.55): # on éclaircit une colormap plus foncée (0.55 trop éclairci)
        r, g, b, a = mcolors.to_rgba(color)
        r = r + (1 - r) * amount
        g = g + (1 - g) * amount
        b = b + (1 - b) * amount
        return (r, g, b, a)

    cmap_units = plt.get_cmap("tab20")
    unit_colors = [lighten(cmap_units(i % cmap_units.N), amount=0.2) for i in range(n_units)]


# =========================
# 2) Ouvrir brut .dat
# =========================
dat_path = nwb_path.with_suffix(".dat")
mm, n_samples = open_dat_memmap(dat_path, n_channels=N_CHANNELS, dtype=DTYPE) # matrice dat,  nb de samples d'un canal
dur_dat = n_samples / FS
print(f"DAT: {dat_path.name} | n_samples={n_samples} | durée~{dur_dat:.2f}s | n_channels={N_CHANNELS}")

# Vérif temps
t_start = max(0.0, float(t_start))
t_end = min(float(t_end), float(dur_dat))
if t_end <= t_start:
    raise ValueError(f"t_end ({t_end}) doit être > t_start ({t_start}).")

# Canaux de la tétrode
raw_ch = tetrode_channels(tetrode_id)
print(f"Canaux bruts tétrode {tetrode_id}: {raw_ch}")

def precompute_minmax_envelope(mm, fs, t_start, t_end, channels, ds_fs=1000):
    """
    Produit une enveloppe min/max à fréquence ds_fs (Hz) sur [t_start, t_end].
    Retour:
      t_ds (sec absolues), ymin (n_pts, n_ch), ymax (n_pts, n_ch)
    """
    s0 = int(np.floor(t_start * fs))
    s1 = int(np.floor(t_end * fs))
    s0 = max(0, s0); s1 = min(mm.shape[0], s1)
    if s1 <= s0:
        raise ValueError("Intervalle brut vide (vérifie t_start/t_end).")

    # taille de bloc en samples
    block = max(1, int(round(fs / ds_fs)))
    n = s1 - s0
    n_blocks = n // block

    # on tronque à un multiple de block pour reshape
    n_use = n_blocks * block
    X = mm[s0:s0+n_use, channels].astype(np.float32)  # (n_use, n_ch)
    X = X.reshape(n_blocks, block, len(channels))

    ymin = np.min(X, axis=1)
    ymax = np.max(X, axis=1)
    # temps au centre de chaque bloc
    t_ds = t_start + (np.arange(n_blocks, dtype=np.float32) + 0.5) * (block / fs)
    return t_ds, ymin, ymax, block

# ex: downsampling a 1000 Hz d'affichage (ok aussi pour 500 ou 2000)
DS_FS = 1000
HILITE_MS = 6
HILITE_HALF_PTS = max(1, int(round(HILITE_MS * DS_FS / 1000.0)))

t_ds, y_min, y_max, block = precompute_minmax_envelope(mm, FS, t_start, t_end, raw_ch, ds_fs=DS_FS)
print(f"RAW downsample précomputé: {DS_FS} Hz | points={len(t_ds)} | block={block} samples")

# ---- gain/scale RAW fixe sur tout l'intervalle (stable) ----
def estimate_global_scale(mm, fs, t_start, t_end, channels, n_probe=80_000, pct=99.0):
    # on prend quelques samples répartis uniformément sur l'intervalle
    s0 = int(t_start * fs)
    s1 = int(t_end * fs)
    s0 = max(0, s0); s1 = min(mm.shape[0], s1)
    if s1 <= s0:
        return 1.0

    n_total = s1 - s0
    step = max(1, n_total // n_probe)
    seg = mm[s0:s1:step, channels].astype(np.float32)
    val = np.nanpercentile(np.abs(seg), pct)
    if not np.isfinite(val) or val <= 0:
        val = 1.0
    return float(val)

S_RAW = estimate_global_scale(mm, FS, t_start, t_end, raw_ch, n_probe=120_000, pct=ROBUST_PCT)
print(f"RAW scale (p{ROBUST_PCT}) = {S_RAW:.2f} (int16 units)")

# =========================
# 3) Frame: frame_t is list of instants of video. Two indices of frame_t are separated in time by dt
# =========================
duration = float(t_end - t_start)
dt_video = speed * (1.0 / fps)        # le pas temporel (en secondes ABSOLUES par frame)
frame_t = t_start + np.arange(0.0, duration, dt_video)
frame_t = frame_t[frame_t <= t_end]

# =========================
# 4) Figure: RAW (signal raw, haut) + RAS (raster, bas)
# =========================
fig = plt.figure(figsize=(10, 6), dpi=300)
gs = fig.add_gridspec(2, 1, height_ratios=[1, 1.5], hspace=0.03)
ax_raw = fig.add_subplot(gs[0])
ax_ras = fig.add_subplot(gs[1], sharex=ax_raw)
fig.patch.set_facecolor(BG)
ax_raw.set_facecolor(BG)
ax_ras.set_facecolor(BG)

def setup_axes():
    for ax in (ax_raw, ax_ras):
        ax.set_xlim(-window, +window)
        ax.get_xaxis().set_visible(False)
        ax.get_yaxis().set_visible(False)
        ax.spines[['right', 'top', 'left', 'bottom']].set_visible(False)
        ax.set_facecolor(BG)
    ax_ras.set_ylim(-0.5, n_units - 0.5)
    ax_ras.grid(True, axis="x", alpha=GRID_ALPHA, color=FG)

setup_axes()
ax_raw.set_ylim(-1.5, (len(raw_ch)-1)*RAW_OFFSET_MULT + 1.5) # Fix Y limits (important sinon autoscale coupe les canaux)

# --- RAW: 4 "lignes" recolorables via LineCollection (1 par canal) ---
raw_collections = []
for ci in range(len(raw_ch)):
    lc = LineCollection([], linewidths=RAW_LINE_W)
    lc.set_alpha(0.95)
    ax_raw.add_collection(lc)
    raw_collections.append(lc)

# (optionnel) on évite cla() et on garde un texte mis à jour
time_text = ax_raw.text(0.01, 1.1, "", transform=ax_raw.transAxes, va="top", color=FG)
ax_raw.axvline(0.0, linestyle="--", linewidth=1.0, alpha=0.35, color=FG)


# =========================
# 5) Update
# =========================
def update(k):
    t = float(frame_t[k])
    t0 = t - window
    t1 = t + window
    lo = max(0.0, t0)

    # ---- RAW ----
    ax_raw.set_xlim(-window, 0.0)
    ax_raw.set_facecolor(BG)
    # (si besoin, assure une fois au début; sinon tu peux laisser)
    ax_raw.get_xaxis().set_visible(False)
    ax_raw.get_yaxis().set_visible(False)
    ax_raw.spines[['right', 'top', 'left', 'bottom']].set_visible(False)
    time_text.set_text(f"t = {t:0.1f}s")

    # Sélection indices signal sous-échantillonné dans la nouvelle fenêtre temporelle
    i0 = np.searchsorted(t_ds, t0, side="left")
    i1 = np.searchsorted(t_ds, t1, side="right")
    x_rel = t_ds[i0:i1] - t

    segmin = y_min[i0:i1, :] / S_RAW
    segmax = y_max[i0:i1, :] / S_RAW
    npts = segmin.shape[0]

    offset = RAW_OFFSET_MULT

    # Prépare une recoloration par segment:
    # on trace en "zigzag" min/max -> on a 2*npts points, donc (2*npts - 1) segments
    if npts >= 2:
        # base color = blanc (ou FG)
        base_rgba = np.array(plt.matplotlib.colors.to_rgba(RAW_COLOR), dtype=np.float32)
        base_rgba[3] = RAW_ALPHA

        # empilement vertical des 4 canaux
        # Pour chaque canal, on construit points/segments une fois, puis on colore certains segments
        for ci in range(segmin.shape[1]):
            y = np.empty(npts * 2, dtype=np.float32)
            y[0::2] = segmin[:, ci] + ci * offset
            y[1::2] = segmax[:, ci] + ci * offset
            x = np.repeat(x_rel, 2)

            points = np.column_stack([x, y]).astype(np.float32)                 # (2*npts, 2)
            segments = np.stack([points[:-1], points[1:]], axis=1)              # (2*npts-1, 2, 2)

            colors = np.tile(base_rgba, (segments.shape[0], 1))                 # (nseg, 4)

            # --- recoloration "dans" la courbe autour des spikes ---
            # largeur en points DS autour du spike
            H = HILITE_HALF_PTS

            for ui, st in enumerate(spike_times):
                if st.size == 0:
                    continue
                a = np.searchsorted(st, t - window, side="left")
                b = np.searchsorted(st, t, side="right")
                if b <= a:
                    continue

                spike_rgba = np.array(plt.matplotlib.colors.to_rgba(unit_colors[ui]), dtype=np.float32)
                spike_rgba[3] = 1.0

                # indices locaux j (sur t_ds[i0:i1])
                # on projette les spikes en indices dans t_ds puis on recadre à [0, npts)
                jg = np.searchsorted(t_ds, st[a:b], side="left")  # indices globaux
                j = jg - i0
                j = j[(j >= 0) & (j < npts)]
                if j.size == 0:
                    continue

                for jj in j:
                    j0 = max(0, int(jj) - H)
                    j1 = min(npts, int(jj) + H + 1)
                    if j1 - j0 < 2:
                        continue

                    # Mapping points->segments:
                    # points zigzag: index_p = 2*k ou 2*k+1
                    # segments index between consecutive points => segment indices couvrent [2*j0 .. 2*j1-2]
                    s0 = 2 * j0
                    s1 = 2 * j1 - 2
                    s0 = max(0, s0)
                    s1 = min(colors.shape[0] - 1, s1)
                    colors[s0:s1+1, :] = spike_rgba

            # applique à la collection du canal
            raw_collections[ci].set_segments(segments)
            raw_collections[ci].set_color(colors)

    else:
        # fenêtre trop petite => vide
        for lc in raw_collections:
            lc.set_segments([])


    # ---- RASTER ----
    # Clear window + style
    ax_ras.cla()
    ax_ras.set_xlim(-window, +window)
    ax_ras.set_ylim(-0.5, n_units - 0.5)
    ax_ras.get_xaxis().set_visible(False)
    ax_ras.get_yaxis().set_visible(False)
    ax_ras.spines[['right', 'top', 'left', 'bottom']].set_visible(False)
    ax_ras.set_facecolor(BG)
    ax_ras.grid(True, axis="x", alpha=GRID_ALPHA, color=FG)

    # masque la moitié droite (future) : rectangle BG sur [0, +window]
    ax_ras.axvspan(0.0, window, color=BG, alpha=1.0, zorder=10)


    # pour chaque unit, on affiche tous les spikes de la fenetre [l0, t1]
    lo_ras = t - window
    hi_ras = t

    for i, st in enumerate(spike_times):
        if st.size == 0:
            continue
        a = np.searchsorted(st, lo_ras, side="left")
        b = np.searchsorted(st, hi_ras, side="right")
        if b > a:
            x = st[a:b] - t   # dans [-window, 0]
            ax_ras.vlines(x, i - HALF_H, i + HALF_H, linewidth=LINE_W, color=unit_colors[i], alpha=0.95, zorder=5)

    ax_ras.axvline(0.0, linestyle="--", linewidth=1.0, alpha=0.35, color=FG, zorder=20)
    return []

anim = FuncAnimation(fig, update, frames=len(frame_t), interval=1000 / fps, blit=False)


# =========================
# 6) Export vidéo + audio mux
# =========================
out_mp4 = Path(f"raw+raster_{patient}_stim{sess}_tt{int(round(tetrode_id))}_{t_start:.0f}-{t_end:.0f}s_middle.mp4")
tmp_video = out_mp4.with_name(out_mp4.stem + "_silent.mp4")
tmp_wav = out_mp4.with_suffix(".wav")

writer = FFMpegWriter(fps=fps, bitrate=4000)

print(f"Export vidéo silencieuse -> {tmp_video}")
anim.save(str(tmp_video), writer=writer)
plt.close(fig)

print("Synth audio...")
audio = synth_morse_audio_from_spikes(
    spike_times=spike_times,
    t_start=t_start,
    t_end=t_end,
    speed=speed,
    sr=SR,
    click_dur=CLICK_DUR,
    base_freq=BASE_FREQ,
    freq_step=FREQ_STEP,
)
write_wav(tmp_wav, audio, sr=SR)

print("Mux audio+vidéo...")
mux_audio_video_ffmpeg(tmp_video, tmp_wav, out_mp4)
print(f"✅ MP4 final -> {out_mp4}")

# Nettoyage
try:
    tmp_video.unlink()
    tmp_wav.unlink()
except Exception:
    pass


NWB path: Spike-sorting/Data_folders/P111_SL64/P111_SL64_stim5/P111_SL64_stim5.nwb
Tétrode 13.0 | units: 17 | keys(ex): [np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17)]
DAT: P111_SL64_stim5.dat | n_samples=152832000 | durée~4664.06s | n_channels=56
Canaux bruts tétrode 13.0: [48, 49, 50, 51]
RAW downsample précomputé: 1000 Hz | points=14894 | block=33 samples
RAW scale (p99.0) = 655.00 (int16 units)
Export vidéo silencieuse -> raw+raster_P111_SL64_stim5_tt13_2160-2175s_middle_silent.mp4
Synth audio...
Mux audio+vidéo...


ffmpeg version 7.1.3 Copyright (c) 2000-2025 the FFmpeg developers
  built with gcc 15.2.0 (GCC)
  configuration: --prefix=/usr --libdir=/usr/lib/x86_64-linux-gnu --optflags='-O2 -pipe -g -Wp,-D_FORTIFY_SOURCE=3 -Wp,-D_GLIBCXX_ASSERTIONS -fexceptions -fstack-protector-strong -grecord-gcc-switches -fasynchronous-unwind-tables -fstack-clash-protection -fcf-protection -fno-omit-frame-pointer -mno-omit-leaf-frame-pointer ' --extra-ldflags='-Wl,-z,relro,-z,now -Wl,--as-needed ' --disable-stripping --disable-doc --disable-static --disable-encoders --disable-decoders --enable-shared --enable-gnutls --enable-gcrypt --enable-ladspa --enable-lcms2 --enable-libaom --enable-libdav1d --enable-libmp3lame --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libjxl --enable-libopus --enable-libpulse --enable-libspeex --enable-libtheora --enable-libvorbis --enable-libvpx --enable-librsvg --enable-libwebp --enable-libxml2 --enable-libopenjpeg --enable-libsvtav1 --enable-openal --enab

✅ MP4 final -> raw+raster_P111_SL64_stim5_tt13_2160-2175s_middle.mp4


[out#0/mp4 @ 0x5603304ec5c0] video:34998KiB audio:85KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.160712%
frame= 1875 fps=0.0 q=-1.0 Lsize=   35139KiB time=00:01:14.92 bitrate=3842.3kbits/s speed= 147x    
[aac @ 0x56033042ea80] Qavg: 63032.227


In [ ]:
# une seule tetrode + raster + EEG brut + bande-son + t_spk réinjectés dans EEG

import os
import wave
import subprocess
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.collections import LineCollection
import pynapple as nap
from pynapple.io.interface_nwb import NWBFile


# =========================
# Paramètres
# =========================
t_start, t_end = 3, 7        # secondes (temps absolu)
patient = "P111_SL64"
sess = "5"
tetrode_id = 13.0               # 1..14 (float)

window = 2 # secondes
speed = 0.2 # 0.3 s pour window de 4 s 
fps = 25

# Raster style
LINE_W = 1.6
HALF_H = 0.45

# Brut .dat
FS = 32768.0
DTYPE = np.int16
N_TETRODES = 14
N_CHANNELS = N_TETRODES * 4  # 56

# Affichage brut
DARK = True
BG = "#0b0f14"  if DARK else "white" # noir bleuté (moins dur que #000)
FG = "#c7d0d9" if DARK else "black" # texte/grille doux
RAW_COLOR = "#545f69" #FG          # même couleur pour tous les canaux
RAW_ALPHA = 0.95
GRID_ALPHA = 0.10
RAW_LINE_W = 0.2
RAW_DOWNSAMPLE_TARGET = 10000   # points max affichés par frame (perfs)

# autoscale robuste
USE_ROBUST_AUTOSCALE = True
ROBUST_PCT = 99.0       # percentile pour échelle
RAW_OFFSET_MULT = 2.5    # offset vertical entre canaux (en unités int16)

# Audio
SR = 44100
CLICK_DUR = 0.020
BASE_FREQ = 450.0
FREQ_STEP = 70.0


# =========================
# Load NWB
# =========================
def get_nwb(patient, session, root="/media/aube/Aube/"):
    path_folder = root + "Spike-sorting/Data_folders/" + patient + "/" + patient + "_stim" + session + "/"
    files_basename = patient + "_stim" + session
    nwbfile_path = path_folder + files_basename + ".nwb"
    print("NWB path:", nwbfile_path)

    if not os.path.exists(nwbfile_path):
        from datetime import datetime
        from dateutil import tz
        from neuroconv.datainterfaces import NeuroScopeSortingInterface

        xml_path = Path(path_folder + files_basename + ".xml")
        interface = NeuroScopeSortingInterface(folder_path=Path(path_folder), xml_file_path=xml_path, verbose=False)
        metadata = interface.get_metadata()
        session_start_time = datetime(2023, 4, 4, 12, 30, 0, tzinfo=tz.gettz("US/Pacific"))
        metadata["NWBFile"].update(session_start_time=session_start_time)
        interface.run_conversion(nwbfile_path=nwbfile_path, metadata=metadata)

    return NWBFile(nwbfile_path)["units"], Path(nwbfile_path)


# =========================
# Outils brut .dat (memmap)
# =========================
def open_dat_memmap(dat_path: Path, n_channels: int, dtype=np.int16):
    if not dat_path.exists():
        raise FileNotFoundError(f".dat introuvable: {dat_path.resolve()}")

    bytes_per_val = np.dtype(dtype).itemsize
    n_vals = dat_path.stat().st_size // bytes_per_val
    if n_vals % n_channels != 0:
        raise ValueError(
            f"Taille .dat incompatible: n_vals={n_vals} n'est pas multiple de n_channels={n_channels}."
        )
    n_samples = n_vals // n_channels
    mm = np.memmap(dat_path, dtype=dtype, mode="r", shape=(n_samples, n_channels), order="C")
    return mm, n_samples

def tetrode_channels(tetrode_id):
    # tetrode_id peut être float -> cast int
    tt = int(round(float(tetrode_id)))
    if tt < 1 or tt > N_TETRODES:
        raise ValueError(f"tetrode_id={tetrode_id} invalide. Attendu: 1..{N_TETRODES}.")
    start = (tt - 1) * 4
    return list(range(start, start + 4))

def get_raw_window(mm, fs, t0, t1, channels, t_min_valid=0.0, target_points=2500):
    """
    Fenêtre [t0, t1] en temps ABSOLU, avec padding NaN si t0 < t_min_valid.
    Retourne x_rel in [-window, 0], seg shape (n_points, n_ch) en float32.
    """
    n_samples_total = mm.shape[0]
    n_ch = len(channels)

    # indices en samples (peuvent être <0)
    s0 = int(np.floor(t0 * fs))
    s1 = int(np.floor(t1 * fs))

    s_min = int(np.floor(t_min_valid * fs))

    # longueur cible en samples (fenêtre constante)
    n_target = max(1, int(np.round((t1 - t0) * fs)))

    # padding à gauche si on est avant t_min_valid
    left_pad = max(0, s_min - s0)

    # clamp lecture
    s0_read = max(s_min, s0)
    s1_read = min(n_samples_total, s1)

    if s1_read > s0_read:
        seg = mm[s0_read:s1_read, channels].astype(np.float32)
    else:
        seg = np.empty((0, n_ch), dtype=np.float32)

    # ajoute padding NaN à gauche
    if left_pad > 0:
        seg = np.vstack([np.full((left_pad, n_ch), np.nan, dtype=np.float32), seg])

    # ajuste à n_target (pad NaN à droite si besoin)
    if seg.shape[0] > n_target:
        seg = seg[:n_target]
    elif seg.shape[0] < n_target:
        pad = n_target - seg.shape[0]
        seg = np.vstack([seg, np.full((pad, n_ch), np.nan, dtype=np.float32)])

    # downsample pour affichage
    n = seg.shape[0]
    if n > target_points:
        step = int(np.ceil(n / target_points))
        seg = seg[::step, :]
        # x_rel recalculé avec step
        x_rel = (np.arange(seg.shape[0], dtype=np.float32) * (step / fs)) - (t1 - t0)
    else:
        x_rel = (np.arange(n, dtype=np.float32) / fs) - (t1 - t0)

    return x_rel, seg

# =========================
# Audio (morse)
# =========================
def synth_morse_audio_from_spikes(spike_times, t_start, t_end, speed, sr=44100,
                                  click_dur=0.020, base_freq=450.0, freq_step=70.0):
    duration_video = float(t_end - t_start) / float(speed)
    n = int(np.ceil(duration_video * sr))
    audio = np.zeros(n, dtype=np.float32)

    L = max(1, int(click_dur * sr))
    tt = np.arange(L, dtype=np.float32) / sr

    env = np.exp(-tt * 45.0).astype(np.float32)
    attack = np.minimum(1.0, tt / 0.002).astype(np.float32)
    env *= attack

    # pour chaque unit, une tonalité
    for i, st in enumerate(spike_times):
        if st.size == 0:
            continue
        f = base_freq + i * freq_step
        tone = (np.sin(2 * np.pi * f * tt) * env).astype(np.float32)

        st_in = st[(st >= t_start) & (st <= t_end)]
        for s in st_in:
            idx0 = int(((float(s) - t_start) / speed) * sr)
            if idx0 < 0 or idx0 >= n:
                continue
            idx1 = min(n, idx0 + L)
            audio[idx0:idx1] += tone[: idx1 - idx0]

    peak = float(np.max(np.abs(audio))) if audio.size else 0.0
    if peak > 0:
        audio /= (peak * 1.05)
    return audio

def write_wav(path, audio, sr=44100):
    audio_i16 = np.int16(np.clip(audio, -1.0, 1.0) * 32767)
    with wave.open(str(path), "wb") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(sr)
        wf.writeframes(audio_i16.tobytes())

def mux_audio_video_ffmpeg(video_in, audio_in, video_out):
    cmd = [
        "ffmpeg", "-y",
        "-i", str(video_in),
        "-i", str(audio_in),
        "-c:v", "copy",
        "-c:a", "aac",
        "-shortest",
        str(video_out),
    ]
    subprocess.run(cmd, check=True)


# =========================
# 1) Charger spikes, filtrer tétrode
# =========================
spk, nwb_path = get_nwb(patient, sess)

epoch_scroll = nap.IntervalSet(start=[t_start], end=[t_end], time_units="s")
spk = spk.restrict(epoch_scroll)

groups_dict = spk.getby_category("group")
available = sorted([float(g) for g in groups_dict.keys()])
if float(tetrode_id) not in [float(g) for g in groups_dict.keys()]:
    raise ValueError(
        f"Aucune unit pour tetrode/group={tetrode_id}. Groupes dispo (intervalle): {available}"
    )

idx_keep = np.asarray(groups_dict[float(tetrode_id)].index, dtype=int)
spk = spk[idx_keep]
unit_keys = list(spk.index)

spike_times = []
for k in unit_keys:
    spike_times.append(np.sort(np.asarray(spk[k].t, dtype=float)))

n_units = len(spike_times)
print(f"Tétrode {tetrode_id} | units: {n_units} | keys(ex): {unit_keys[:8]}")

# Couleurs par unit
if n_units < 12:
    cmap_units = plt.get_cmap("Set3")  # très lisible sur fond sombre #plt.get_cmap("tab20")
    unit_colors = [cmap_units(i % cmap_units.N) for i in range(n_units)]
else:
    import matplotlib.colors as mcolors
    def lighten(color, amount=0.55): # on éclaircit une colormap plus foncée (0.55 trop éclairci)
        r, g, b, a = mcolors.to_rgba(color)
        r = r + (1 - r) * amount
        g = g + (1 - g) * amount
        b = b + (1 - b) * amount
        return (r, g, b, a)

    cmap_units = plt.get_cmap("tab20")
    unit_colors = [lighten(cmap_units(i % cmap_units.N), amount=0.2) for i in range(n_units)]


# =========================
# 2) Ouvrir brut .dat
# =========================
dat_path = nwb_path.with_suffix(".dat")
mm, n_samples = open_dat_memmap(dat_path, n_channels=N_CHANNELS, dtype=DTYPE) # matrice dat,  nb de samples d'un canal
dur_dat = n_samples / FS
print(f"DAT: {dat_path.name} | n_samples={n_samples} | durée~{dur_dat:.2f}s | n_channels={N_CHANNELS}")

# Vérif temps
t_start = max(0.0, float(t_start))
t_end = min(float(t_end), float(dur_dat))
if t_end <= t_start:
    raise ValueError(f"t_end ({t_end}) doit être > t_start ({t_start}).")

# Canaux de la tétrode
raw_ch = tetrode_channels(tetrode_id)
print(f"Canaux bruts tétrode {tetrode_id}: {raw_ch}")

def precompute_minmax_envelope(mm, fs, t_start, t_end, channels, ds_fs=1000):
    """
    Produit une enveloppe min/max à fréquence ds_fs (Hz) sur [t_start, t_end].
    Retour:
      t_ds (sec absolues), ymin (n_pts, n_ch), ymax (n_pts, n_ch)
    """
    s0 = int(np.floor(t_start * fs))
    s1 = int(np.floor(t_end * fs))
    s0 = max(0, s0); s1 = min(mm.shape[0], s1)
    if s1 <= s0:
        raise ValueError("Intervalle brut vide (vérifie t_start/t_end).")

    # taille de bloc en samples
    block = max(1, int(round(fs / ds_fs)))
    n = s1 - s0
    n_blocks = n // block

    # on tronque à un multiple de block pour reshape
    n_use = n_blocks * block
    X = mm[s0:s0+n_use, channels].astype(np.float32)  # (n_use, n_ch)
    X = X.reshape(n_blocks, block, len(channels))

    ymin = np.min(X, axis=1)
    ymax = np.max(X, axis=1)
    # temps au centre de chaque bloc
    t_ds = t_start + (np.arange(n_blocks, dtype=np.float32) + 0.5) * (block / fs)
    return t_ds, ymin, ymax, block

# ex: downsampling a 1000 Hz d'affichage (ok aussi pour 500 ou 2000)
DS_FS = 1000
HILITE_MS = 6
HILITE_HALF_PTS = max(1, int(round(HILITE_MS * DS_FS / 1000.0)))

t_ds, y_min, y_max, block = precompute_minmax_envelope(mm, FS, t_start, t_end, raw_ch, ds_fs=DS_FS)
print(f"RAW downsample précomputé: {DS_FS} Hz | points={len(t_ds)} | block={block} samples")

# ---- gain/scale RAW fixe sur tout l'intervalle (stable) ----
def estimate_global_scale(mm, fs, t_start, t_end, channels, n_probe=80_000, pct=99.0):
    # on prend quelques samples répartis uniformément sur l'intervalle
    s0 = int(t_start * fs)
    s1 = int(t_end * fs)
    s0 = max(0, s0); s1 = min(mm.shape[0], s1)
    if s1 <= s0:
        return 1.0

    n_total = s1 - s0
    step = max(1, n_total // n_probe)
    seg = mm[s0:s1:step, channels].astype(np.float32)
    val = np.nanpercentile(np.abs(seg), pct)
    if not np.isfinite(val) or val <= 0:
        val = 1.0
    return float(val)

S_RAW = estimate_global_scale(mm, FS, t_start, t_end, raw_ch, n_probe=120_000, pct=ROBUST_PCT)
print(f"RAW scale (p{ROBUST_PCT}) = {S_RAW:.2f} (int16 units)")

# =========================
# 3) Frame: frame_t is list of instants of video. Two indices of frame_t are separated in time by dt
# =========================
duration = float(t_end - t_start)
dt_video = speed * (1.0 / fps)        # le pas temporel (en secondes ABSOLUES par frame)
frame_t = t_start + np.arange(0.0, duration, dt_video)
frame_t = frame_t[frame_t <= t_end]

# =========================
# 4) Figure: RAW (signal raw, haut) + RAS (raster, bas)
# =========================
fig = plt.figure(figsize=(10, 6), dpi=300)
gs = fig.add_gridspec(2, 1, height_ratios=[1, 1.5], hspace=0.03)
ax_raw = fig.add_subplot(gs[0])
ax_ras = fig.add_subplot(gs[1], sharex=ax_raw)
fig.patch.set_facecolor(BG)
ax_raw.set_facecolor(BG)
ax_ras.set_facecolor(BG)

def setup_axes():
    for ax in (ax_raw, ax_ras):
        ax.set_xlim(-window, 0.0)
        ax.get_xaxis().set_visible(False)
        ax.get_yaxis().set_visible(False)
        ax.spines[['right', 'top', 'left', 'bottom']].set_visible(False)
        ax.set_facecolor(BG)
    ax_ras.set_ylim(-0.5, n_units - 0.5)
    ax_ras.grid(True, axis="x", alpha=GRID_ALPHA, color=FG)

setup_axes()
ax_raw.set_ylim(-1.5, (len(raw_ch)-1)*RAW_OFFSET_MULT + 1.5) # Fix Y limits (important sinon autoscale coupe les canaux)

# --- RAW: 4 "lignes" recolorables via LineCollection (1 par canal) ---
raw_collections = []
for ci in range(len(raw_ch)):
    lc = LineCollection([], linewidths=RAW_LINE_W)
    lc.set_alpha(0.95)
    ax_raw.add_collection(lc)
    raw_collections.append(lc)

# (optionnel) on évite cla() et on garde un texte mis à jour
time_text = ax_raw.text(0.01, 1.1, "", transform=ax_raw.transAxes, va="top", color=FG)
ax_raw.axvline(0.0, linestyle="--", linewidth=1.0, alpha=0.35, color=FG)


# =========================
# 5) Update
# =========================
def update(k):
    t = float(frame_t[k])
    t0 = t - window
    t1 = t
    lo = max(0.0, t0)

    # ---- RAW ----
    ax_raw.set_xlim(-window, 0.0)
    ax_raw.set_facecolor(BG)
    # (si besoin, assure une fois au début; sinon tu peux laisser)
    ax_raw.get_xaxis().set_visible(False)
    ax_raw.get_yaxis().set_visible(False)
    ax_raw.spines[['right', 'top', 'left', 'bottom']].set_visible(False)
    time_text.set_text(f"t = {t:0.2f}s")

    # Sélection indices signal sous-échantillonné dans la nouvelle fenêtre temporelle
    i0 = np.searchsorted(t_ds, t0, side="left")
    i1 = np.searchsorted(t_ds, t1, side="right")
    x_rel = t_ds[i0:i1] - t

    segmin = y_min[i0:i1, :] / S_RAW
    segmax = y_max[i0:i1, :] / S_RAW
    npts = segmin.shape[0]

    offset = RAW_OFFSET_MULT

    # Prépare une recoloration par segment:
    # on trace en "zigzag" min/max -> on a 2*npts points, donc (2*npts - 1) segments
    if npts >= 2:
        # base color = blanc (ou FG)
        base_rgba = np.array(plt.matplotlib.colors.to_rgba(RAW_COLOR), dtype=np.float32)
        base_rgba[3] = RAW_ALPHA

        # empilement vertical des 4 canaux
        # Pour chaque canal, on construit points/segments une fois, puis on colore certains segments
        for ci in range(segmin.shape[1]):
            y = np.empty(npts * 2, dtype=np.float32)
            y[0::2] = segmin[:, ci] + ci * offset
            y[1::2] = segmax[:, ci] + ci * offset
            x = np.repeat(x_rel, 2)

            points = np.column_stack([x, y]).astype(np.float32)                 # (2*npts, 2)
            segments = np.stack([points[:-1], points[1:]], axis=1)              # (2*npts-1, 2, 2)

            colors = np.tile(base_rgba, (segments.shape[0], 1))                 # (nseg, 4)

            # --- recoloration "dans" la courbe autour des spikes ---
            # largeur en points DS autour du spike
            H = HILITE_HALF_PTS

            for ui, st in enumerate(spike_times):
                if st.size == 0:
                    continue
                a = np.searchsorted(st, t0, side="left")
                b = np.searchsorted(st, t1, side="right")
                if b <= a:
                    continue

                spike_rgba = np.array(plt.matplotlib.colors.to_rgba(unit_colors[ui]), dtype=np.float32)
                spike_rgba[3] = 1.0

                # indices locaux j (sur t_ds[i0:i1])
                # on projette les spikes en indices dans t_ds puis on recadre à [0, npts)
                jg = np.searchsorted(t_ds, st[a:b], side="left")  # indices globaux
                j = jg - i0
                j = j[(j >= 0) & (j < npts)]
                if j.size == 0:
                    continue

                for jj in j:
                    j0 = max(0, int(jj) - H)
                    j1 = min(npts, int(jj) + H + 1)
                    if j1 - j0 < 2:
                        continue

                    # Mapping points->segments:
                    # points zigzag: index_p = 2*k ou 2*k+1
                    # segments index between consecutive points => segment indices couvrent [2*j0 .. 2*j1-2]
                    s0 = 2 * j0
                    s1 = 2 * j1 - 2
                    s0 = max(0, s0)
                    s1 = min(colors.shape[0] - 1, s1)
                    colors[s0:s1+1, :] = spike_rgba

            # applique à la collection du canal
            raw_collections[ci].set_segments(segments)
            raw_collections[ci].set_color(colors)

    else:
        # fenêtre trop petite => vide
        for lc in raw_collections:
            lc.set_segments([])


    # ---- RASTER ----
    # Clear window + style
    ax_ras.cla()
    ax_ras.set_xlim(-window, 0.0)
    ax_ras.set_ylim(-0.5, n_units - 0.5)
    ax_ras.get_xaxis().set_visible(False)
    ax_ras.get_yaxis().set_visible(False)
    ax_ras.spines[['right', 'top', 'left', 'bottom']].set_visible(False)
    ax_ras.set_facecolor(BG)
    ax_ras.grid(True, axis="x", alpha=0.15, color=FG)

    # pour chaque unit, on affiche tous les spikes de la fenetre [l0, t1]
    for i, st in enumerate(spike_times):
        if st.size == 0:
            continue
        a = np.searchsorted(st, lo, side="left")
        b = np.searchsorted(st, t1, side="right")
        if b > a:
            x = st[a:b] - t
            ax_ras.vlines(x, i - HALF_H, i + HALF_H, linewidth=LINE_W, color=unit_colors[i], alpha=0.95)

    ax_ras.axvline(0.0, linestyle="--", linewidth=1.0, alpha=0.35, color=FG)
    return []

anim = FuncAnimation(fig, update, frames=len(frame_t), interval=1000 / fps, blit=False)


# =========================
# 6) Export vidéo + audio mux
# =========================
out_mp4 = Path(f"raw+raster_{patient}_stim{sess}_tt{int(round(tetrode_id))}_{t_start:.0f}-{t_end:.0f}s.mp4")
tmp_video = out_mp4.with_name(out_mp4.stem + "_silent.mp4")
tmp_wav = out_mp4.with_suffix(".wav")

writer = FFMpegWriter(fps=fps, bitrate=4000)

print(f"Export vidéo silencieuse -> {tmp_video}")
anim.save(str(tmp_video), writer=writer)
plt.close(fig)

print("Synth audio...")
audio = synth_morse_audio_from_spikes(
    spike_times=spike_times,
    t_start=t_start,
    t_end=t_end,
    speed=speed,
    sr=SR,
    click_dur=CLICK_DUR,
    base_freq=BASE_FREQ,
    freq_step=FREQ_STEP,
)
write_wav(tmp_wav, audio, sr=SR)

print("Mux audio+vidéo...")
mux_audio_video_ffmpeg(tmp_video, tmp_wav, out_mp4)
print(f"✅ MP4 final -> {out_mp4}")

# Nettoyage
try:
    tmp_video.unlink()
    tmp_wav.unlink()
except Exception:
    pass


NWB path: /media/aube/Aube/Spike-sorting/Data_folders/P111_SL64/P111_SL64_stim5/P111_SL64_stim5.nwb
Tétrode 13.0 | units: 17 | keys(ex): [np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17)]
DAT: P111_SL64_stim5.dat | n_samples=152832000 | durée~4664.06s | n_channels=56
Canaux bruts tétrode 13.0: [48, 49, 50, 51]
RAW downsample précomputé: 1000 Hz | points=3971 | block=33 samples
RAW scale (p99.0) = 208.00 (int16 units)
Export vidéo silencieuse -> raw+raster_P111_SL64_stim5_tt13_3-7s_silent.mp4
Synth audio...
Mux audio+vidéo...


ffmpeg version 7.1.3 Copyright (c) 2000-2025 the FFmpeg developers
  built with gcc 15.2.0 (GCC)
  configuration: --prefix=/usr --libdir=/usr/lib/x86_64-linux-gnu --optflags='-O2 -pipe -g -Wp,-D_FORTIFY_SOURCE=3 -Wp,-D_GLIBCXX_ASSERTIONS -fexceptions -fstack-protector-strong -grecord-gcc-switches -fasynchronous-unwind-tables -fstack-clash-protection -fcf-protection -fno-omit-frame-pointer -mno-omit-leaf-frame-pointer ' --extra-ldflags='-Wl,-z,relro,-z,now -Wl,--as-needed ' --disable-stripping --disable-doc --disable-static --disable-encoders --disable-decoders --enable-shared --enable-gnutls --enable-gcrypt --enable-ladspa --enable-lcms2 --enable-libaom --enable-libdav1d --enable-libmp3lame --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libjxl --enable-libopus --enable-libpulse --enable-libspeex --enable-libtheora --enable-libvorbis --enable-libvpx --enable-librsvg --enable-libwebp --enable-libxml2 --enable-libopenjpeg --enable-libsvtav1 --enable-openal --enab

✅ MP4 final -> raw+raster_P111_SL64_stim5_tt13_3-7s.mp4


[out#0/mp4 @ 0x5559304bd780] video:11815KiB audio:34KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.142877%
frame=  500 fps=0.0 q=-1.0 Lsize=   11866KiB time=00:00:19.92 bitrate=4879.9kbits/s speed= 103x    
[aac @ 0x55593035ba80] Qavg: 62206.566


In [ ]:
# une seule tetrode + raster + EEG brut + bande-son. Sans recoloration signal.

import os
import wave
import subprocess
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
import pynapple as nap
from pynapple.io.interface_nwb import NWBFile


# =========================
# Paramètres
# =========================
t_start, t_end = 3, 7        # secondes (temps absolu)
patient = "P111_SL64"
sess = "5"
tetrode_id = 13.0               # 1..14 (float)

window = 4.0
speed = 0.3
fps = 25

# Raster style
LINE_W = 1.6
HALF_H = 0.45

# Brut .dat
FS = 32768.0
DTYPE = np.int16
N_TETRODES = 14
N_CHANNELS = N_TETRODES * 4  # 56

# Affichage brut
DARK = True
BG = "black" if DARK else "white"
FG = "white" if DARK else "black"
RAW_COLOR = FG          # même couleur pour tous les canaux
RAW_LINE_W = 0.4
RAW_DOWNSAMPLE_TARGET = 2500   # points max affichés par frame (perfs)

# autoscale robuste
USE_ROBUST_AUTOSCALE = True
ROBUST_PCT = 99.0       # percentile pour échelle
RAW_OFFSET_MULT = 2.5    # offset vertical entre canaux (en unités int16)

# Audio
SR = 44100
CLICK_DUR = 0.020
BASE_FREQ = 450.0
FREQ_STEP = 70.0


# =========================
# Loader NWB (le tien)
# =========================
def get_nwb(patient, session, root="/media/aube/Aube/"):
    path_folder = root + "Spike-sorting/Data_folders/" + patient + "/" + patient + "_stim" + session + "/"
    files_basename = patient + "_stim" + session
    nwbfile_path = path_folder + files_basename + ".nwb"
    print("NWB path:", nwbfile_path)

    if not os.path.exists(nwbfile_path):
        from datetime import datetime
        from dateutil import tz
        from neuroconv.datainterfaces import NeuroScopeSortingInterface

        xml_path = Path(path_folder + files_basename + ".xml")
        interface = NeuroScopeSortingInterface(folder_path=Path(path_folder), xml_file_path=xml_path, verbose=False)
        metadata = interface.get_metadata()
        session_start_time = datetime(2023, 4, 4, 12, 30, 0, tzinfo=tz.gettz("US/Pacific"))
        metadata["NWBFile"].update(session_start_time=session_start_time)
        interface.run_conversion(nwbfile_path=nwbfile_path, metadata=metadata)

    return NWBFile(nwbfile_path)["units"], Path(nwbfile_path)


# =========================
# Outils brut .dat (memmap)
# =========================
def open_dat_memmap(dat_path: Path, n_channels: int, dtype=np.int16):
    if not dat_path.exists():
        raise FileNotFoundError(f".dat introuvable: {dat_path.resolve()}")

    bytes_per_val = np.dtype(dtype).itemsize
    n_vals = dat_path.stat().st_size // bytes_per_val
    if n_vals % n_channels != 0:
        raise ValueError(
            f"Taille .dat incompatible: n_vals={n_vals} n'est pas multiple de n_channels={n_channels}."
        )
    n_samples = n_vals // n_channels
    mm = np.memmap(dat_path, dtype=dtype, mode="r", shape=(n_samples, n_channels), order="C")
    return mm, n_samples

def tetrode_channels(tetrode_id):
    # tetrode_id peut être float -> cast int
    tt = int(round(float(tetrode_id)))
    if tt < 1 or tt > N_TETRODES:
        raise ValueError(f"tetrode_id={tetrode_id} invalide. Attendu: 1..{N_TETRODES}.")
    start = (tt - 1) * 4
    return list(range(start, start + 4))

def get_raw_window(mm, fs, t0, t1, channels, t_min_valid=0.0, target_points=2500):
    """
    Fenêtre [t0, t1] en temps ABSOLU, avec padding NaN si t0 < t_min_valid.
    Retourne x_rel in [-window, 0], seg shape (n_points, n_ch) en float32.
    """
    n_samples_total = mm.shape[0]
    n_ch = len(channels)

    # indices en samples (peuvent être <0)
    s0 = int(np.floor(t0 * fs))
    s1 = int(np.floor(t1 * fs))

    s_min = int(np.floor(t_min_valid * fs))

    # longueur cible en samples (fenêtre constante)
    n_target = max(1, int(np.round((t1 - t0) * fs)))

    # padding à gauche si on est avant t_min_valid
    left_pad = max(0, s_min - s0)

    # clamp lecture
    s0_read = max(s_min, s0)
    s1_read = min(n_samples_total, s1)

    if s1_read > s0_read:
        seg = mm[s0_read:s1_read, channels].astype(np.float32)
    else:
        seg = np.empty((0, n_ch), dtype=np.float32)

    # ajoute padding NaN à gauche
    if left_pad > 0:
        seg = np.vstack([np.full((left_pad, n_ch), np.nan, dtype=np.float32), seg])

    # ajuste à n_target (pad NaN à droite si besoin)
    if seg.shape[0] > n_target:
        seg = seg[:n_target]
    elif seg.shape[0] < n_target:
        pad = n_target - seg.shape[0]
        seg = np.vstack([seg, np.full((pad, n_ch), np.nan, dtype=np.float32)])

    # downsample pour affichage
    n = seg.shape[0]
    if n > target_points:
        step = int(np.ceil(n / target_points))
        seg = seg[::step, :]
        # x_rel recalculé avec step
        x_rel = (np.arange(seg.shape[0], dtype=np.float32) * (step / fs)) - (t1 - t0)
    else:
        x_rel = (np.arange(n, dtype=np.float32) / fs) - (t1 - t0)

    return x_rel, seg

# =========================
# Audio (morse)
# =========================
def synth_morse_audio_from_spikes(spike_times, t_start, t_end, speed, sr=44100,
                                  click_dur=0.020, base_freq=450.0, freq_step=70.0):
    duration_video = float(t_end - t_start) / float(speed)
    n = int(np.ceil(duration_video * sr))
    audio = np.zeros(n, dtype=np.float32)

    L = max(1, int(click_dur * sr))
    tt = np.arange(L, dtype=np.float32) / sr

    env = np.exp(-tt * 45.0).astype(np.float32)
    attack = np.minimum(1.0, tt / 0.002).astype(np.float32)
    env *= attack

    for i, st in enumerate(spike_times):
        if st.size == 0:
            continue
        f = base_freq + i * freq_step
        tone = (np.sin(2 * np.pi * f * tt) * env).astype(np.float32)

        st_in = st[(st >= t_start) & (st <= t_end)]
        for s in st_in:
            idx0 = int(((float(s) - t_start) / speed) * sr)
            if idx0 < 0 or idx0 >= n:
                continue
            idx1 = min(n, idx0 + L)
            audio[idx0:idx1] += tone[: idx1 - idx0]

    peak = float(np.max(np.abs(audio))) if audio.size else 0.0
    if peak > 0:
        audio /= (peak * 1.05)
    return audio

def write_wav(path, audio, sr=44100):
    audio_i16 = np.int16(np.clip(audio, -1.0, 1.0) * 32767)
    with wave.open(str(path), "wb") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(sr)
        wf.writeframes(audio_i16.tobytes())

def mux_audio_video_ffmpeg(video_in, audio_in, video_out):
    cmd = [
        "ffmpeg", "-y",
        "-i", str(video_in),
        "-i", str(audio_in),
        "-c:v", "copy",
        "-c:a", "aac",
        "-shortest",
        str(video_out),
    ]
    subprocess.run(cmd, check=True)


# =========================
# 1) Charger spikes, filtrer tétrode
# =========================
spk, nwb_path = get_nwb(patient, sess)

epoch_scroll = nap.IntervalSet(start=[t_start], end=[t_end], time_units="s")
spk = spk.restrict(epoch_scroll)

groups_dict = spk.getby_category("group")
available = sorted([float(g) for g in groups_dict.keys()])
if float(tetrode_id) not in [float(g) for g in groups_dict.keys()]:
    raise ValueError(
        f"Aucune unit pour tetrode/group={tetrode_id}. Groupes dispo (intervalle): {available}"
    )

idx_keep = np.asarray(groups_dict[float(tetrode_id)].index, dtype=int)
spk = spk[idx_keep]
unit_keys = list(spk.index)

spike_times = []
for k in unit_keys:
    spike_times.append(np.sort(np.asarray(spk[k].t, dtype=float)))

n_units = len(spike_times)
print(f"Tétrode {tetrode_id} | units: {n_units} | keys(ex): {unit_keys[:8]}")

# Couleurs par unit
cmap_units = plt.get_cmap("tab20")
unit_colors = [cmap_units(i % cmap_units.N) for i in range(n_units)]


# =========================
# 2) Ouvrir brut .dat
# =========================
dat_path = nwb_path.with_suffix(".dat")
mm, n_samples = open_dat_memmap(dat_path, n_channels=N_CHANNELS, dtype=DTYPE) # matrice dat,  nb de samples d'un canal
dur_dat = n_samples / FS
print(f"DAT: {dat_path.name} | n_samples={n_samples} | durée~{dur_dat:.2f}s | n_channels={N_CHANNELS}")

# Vérif temps
t_start = max(0.0, float(t_start))
t_end = min(float(t_end), float(dur_dat))
if t_end <= t_start:
    raise ValueError(f"t_end ({t_end}) doit être > t_start ({t_start}).")

# Canaux de la tétrode
raw_ch = tetrode_channels(tetrode_id)
print(f"Canaux bruts tétrode {tetrode_id}: {raw_ch}")

def precompute_minmax_envelope(mm, fs, t_start, t_end, channels, ds_fs=1000):
    """
    Produit une enveloppe min/max à fréquence ds_fs (Hz) sur [t_start, t_end].
    Retour:
      t_ds (sec absolues), ymin (n_pts, n_ch), ymax (n_pts, n_ch)
    """
    s0 = int(np.floor(t_start * fs))
    s1 = int(np.floor(t_end * fs))
    s0 = max(0, s0); s1 = min(mm.shape[0], s1)
    if s1 <= s0:
        raise ValueError("Intervalle brut vide (vérifie t_start/t_end).")

    # taille de bloc en samples
    block = max(1, int(round(fs / ds_fs)))
    n = s1 - s0
    n_blocks = n // block

    # on tronque à un multiple de block pour reshape
    n_use = n_blocks * block
    X = mm[s0:s0+n_use, channels].astype(np.float32)  # (n_use, n_ch)
    X = X.reshape(n_blocks, block, len(channels))

    ymin = np.min(X, axis=1)
    ymax = np.max(X, axis=1)
    # temps au centre de chaque bloc
    t_ds = t_start + (np.arange(n_blocks, dtype=np.float32) + 0.5) * (block / fs)
    return t_ds, ymin, ymax, block

# ex: 1000 Hz d'affichage (tu peux mettre 500 ou 2000)
DS_FS = 1000
t_ds, y_min, y_max, block = precompute_minmax_envelope(mm, FS, t_start, t_end, raw_ch, ds_fs=DS_FS)
print(f"RAW downsample précomputé: {DS_FS} Hz | points={len(t_ds)} | block={block} samples")

# ---- gain/scale RAW fixe sur tout l'intervalle (stable) ----
def estimate_global_scale(mm, fs, t_start, t_end, channels, n_probe=80_000, pct=99.0):
    # on prend quelques samples répartis uniformément sur l'intervalle
    s0 = int(t_start * fs)
    s1 = int(t_end * fs)
    s0 = max(0, s0); s1 = min(mm.shape[0], s1)
    if s1 <= s0:
        return 1.0

    n_total = s1 - s0
    step = max(1, n_total // n_probe)
    seg = mm[s0:s1:step, channels].astype(np.float32)
    val = np.nanpercentile(np.abs(seg), pct)
    if not np.isfinite(val) or val <= 0:
        val = 1.0
    return float(val)

S_RAW = estimate_global_scale(mm, FS, t_start, t_end, raw_ch, n_probe=120_000, pct=ROBUST_PCT)
print(f"RAW scale (p{ROBUST_PCT}) = {S_RAW:.2f} (int16 units)")

# =========================
# 3) Frame: frame_t is list of instants of video. Two indices of frame_t are separated in time by dt
# =========================
duration = float(t_end - t_start)
dt_video = speed * (1.0 / fps)        # le pas temporel (en secondes ABSOLUES par frame)
frame_t = t_start + np.arange(0.0, duration, dt_video)
frame_t = frame_t[frame_t <= t_end]

# =========================
# 4) Figure: RAW (signal raw, haut) + RAS (raster, bas)
# =========================
fig = plt.figure(figsize=(10, 6), dpi=120)
gs = fig.add_gridspec(2, 1, height_ratios=[1, 2], hspace=0.03)
ax_raw = fig.add_subplot(gs[0])
ax_ras = fig.add_subplot(gs[1], sharex=ax_raw)
fig.patch.set_facecolor(BG)
ax_raw.set_facecolor(BG)
ax_ras.set_facecolor(BG)

def setup_axes():
    for ax in (ax_raw, ax_ras):
        ax.set_xlim(-window, 0.0)
        ax.get_xaxis().set_visible(False)
        ax.get_yaxis().set_visible(False)
        ax.spines[['right', 'top', 'left', 'bottom']].set_visible(False)
        ax.set_facecolor(BG)
    ax_ras.set_ylim(-0.5, n_units - 0.5)
    ax_ras.grid(True, axis="x", alpha=0.15, color=FG)

setup_axes()

# Lignes raw (4 canaux)
raw_cmap = plt.get_cmap("tab10")
raw_color = raw_cmap(0)
raw_lines = []
for i in range(len(raw_ch)):
    (ln,) = ax_raw.plot([], [], linewidth=RAW_LINE_W, color=raw_color)
    raw_lines.append(ln)

# =========================
# 5) Update
# =========================
def update(k):
    t = float(frame_t[k])
    t0 = t - window
    t1 = t
    lo = max(0.0, t0)

    # ---- RAW ----
    # Clear window + style
    ax_raw.cla()
    ax_raw.set_xlim(-window, 0.0)
    ax_raw.get_xaxis().set_visible(False)
    ax_raw.get_yaxis().set_visible(False)
    ax_raw.spines[['right', 'top', 'left', 'bottom']].set_visible(False)
    ax_raw.set_facecolor(BG)
    ax_raw.axvline(0.0, linestyle="--", linewidth=1.0, alpha=0.6, color=FG)
    ax_raw.text(0.01, 1.1, f"t = {t:0.2f}s", transform=ax_raw.transAxes, va="top", color=FG)

    # Sélection indices signal sous-échantillonné dans la nouvelle fenêtre temporelle
    i0 = np.searchsorted(t_ds, t0, side="left")
    i1 = np.searchsorted(t_ds, t1, side="right")
    x_rel = t_ds[i0:i1] - t      # x relatif

    # Normalisation stable (gain fixe)
    segmin = y_min[i0:i1, :] / S_RAW
    segmax = y_max[i0:i1, :] / S_RAW

    # empilement vertical des 4 canaux
    offset = RAW_OFFSET_MULT
    for ci in range(segmin.shape[1]):
        # on trace une courbe "zigzag" min/max pour conserver les pics
        y = np.empty(segmin.shape[0] * 2, dtype=np.float32)
        y[0::2] = segmin[:, ci] + ci * offset
        y[1::2] = segmax[:, ci] + ci * offset
        x = np.repeat(x_rel, 2)
        ax_raw.plot(x, y, linewidth=RAW_LINE_W, color=RAW_COLOR, alpha=0.95)

    # ---- RASTER ----
    # Clear window + style
    ax_ras.cla()
    ax_ras.set_xlim(-window, 0.0)
    ax_ras.set_ylim(-0.5, n_units - 0.5)
    ax_ras.get_xaxis().set_visible(False)
    ax_ras.get_yaxis().set_visible(False)
    ax_ras.spines[['right', 'top', 'left', 'bottom']].set_visible(False)
    ax_ras.set_facecolor(BG)
    ax_ras.grid(True, axis="x", alpha=0.15, color=FG)

    # pour chaque unit, on affiche tous les spikes de la fenetre [l0, t1]
    for i, st in enumerate(spike_times):
        if st.size == 0:
            continue
        a = np.searchsorted(st, lo, side="left")
        b = np.searchsorted(st, t1, side="right")
        if b > a:
            x = st[a:b] - t
            ax_ras.vlines(x, i - HALF_H, i + HALF_H, linewidth=LINE_W, color=unit_colors[i], alpha=0.95)

    ax_ras.axvline(0.0, linestyle="--", linewidth=1.0, alpha=0.6, color=FG)
    return []

anim = FuncAnimation(fig, update, frames=len(frame_t), interval=1000 / fps, blit=False)


# =========================
# 6) Export vidéo + audio mux
# =========================
out_mp4 = Path(f"raw+raster_{patient}_stim{sess}_{t_start:.0f}-{t_end:.0f}s_tt{int(round(tetrode_id))}.mp4")
tmp_video = out_mp4.with_name(out_mp4.stem + "_silent.mp4")
tmp_wav = out_mp4.with_suffix(".wav")

writer = FFMpegWriter(fps=fps, bitrate=4000)

print(f"Export vidéo silencieuse -> {tmp_video}")
anim.save(str(tmp_video), writer=writer)
plt.close(fig)

print("Synth audio...")
audio = synth_morse_audio_from_spikes(
    spike_times=spike_times,
    t_start=t_start,
    t_end=t_end,
    speed=speed,
    sr=SR,
    click_dur=CLICK_DUR,
    base_freq=BASE_FREQ,
    freq_step=FREQ_STEP,
)
write_wav(tmp_wav, audio, sr=SR)

print("Mux audio+vidéo...")
mux_audio_video_ffmpeg(tmp_video, tmp_wav, out_mp4)
print(f"✅ MP4 final -> {out_mp4}")

# Nettoyage
try:
    tmp_video.unlink()
    tmp_wav.unlink()
except Exception:
    pass


NWB path: /media/aube/Aube/Spike-sorting/Data_folders/P111_SL64/P111_SL64_stim5/P111_SL64_stim5.nwb
Tétrode 13.0 | units: 17 | keys(ex): [np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17)]
DAT: P111_SL64_stim5.dat | n_samples=152832000 | durée~4664.06s | n_channels=56
Canaux bruts tétrode 13.0: [48, 49, 50, 51]
RAW downsample précomputé: 1000 Hz | points=3971 | block=33 samples
RAW scale (p99.0) = 208.00 (int16 units)
Export vidéo silencieuse -> raw+raster_P111_SL64_stim5_3-7s_tt13_silent.mp4
Synth audio...
Mux audio+vidéo...


ffmpeg version 7.1.3 Copyright (c) 2000-2025 the FFmpeg developers
  built with gcc 15.2.0 (GCC)
  configuration: --prefix=/usr --libdir=/usr/lib/x86_64-linux-gnu --optflags='-O2 -pipe -g -Wp,-D_FORTIFY_SOURCE=3 -Wp,-D_GLIBCXX_ASSERTIONS -fexceptions -fstack-protector-strong -grecord-gcc-switches -fasynchronous-unwind-tables -fstack-clash-protection -fcf-protection -fno-omit-frame-pointer -mno-omit-leaf-frame-pointer ' --extra-ldflags='-Wl,-z,relro,-z,now -Wl,--as-needed ' --disable-stripping --disable-doc --disable-static --disable-encoders --disable-decoders --enable-shared --enable-gnutls --enable-gcrypt --enable-ladspa --enable-lcms2 --enable-libaom --enable-libdav1d --enable-libmp3lame --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libjxl --enable-libopus --enable-libpulse --enable-libspeex --enable-libtheora --enable-libvorbis --enable-libvpx --enable-librsvg --enable-libwebp --enable-libxml2 --enable-libopenjpeg --enable-libsvtav1 --enable-openal --enab

✅ MP4 final -> raw+raster_P111_SL64_stim5_3-7s_tt13.mp4


In [ ]:
# une seule tetrode et bande-son. Pas d'EEG.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
import pynapple as nap

t_start, t_end = 0, 20 # en secondes (temps absolu NWB)
patient = 'P111_SL64'
sess = '5'
tetrode_id = 13.0   # <-- choisis ici (ex: 1, 4, 5, 7, 8, 13, 14 ...)


window = 4.0   # largeur de fenêtre affichée
speed = 1.0 # vitesse de défilement relativement au réel
LINE_W = 1.6      # <- épaisseur des traits (teste 1.2, 1.6, 2.0)
HALF_H = 0.45     # <- hauteur des traits (0.4 -> 0.45/0.49)

# -------------------------------------------------
# 1) Charger spikes réels
# -------------------------------------------------

import os
from pathlib import Path
from pynapple.io.interface_nwb import NWBFile

def get_nwb(patient, session, root='/media/aube/Aube/'):
    path_folder = root + 'Spike-sorting/Data_folders/'+patient+'/'+patient+'_stim'+session+'/'
    files_basename = patient+'_stim'+session
    nwbfile_path = path_folder + files_basename + ".nwb"
    print("NWB path:", nwbfile_path)

    if not os.path.exists(nwbfile_path):
        from datetime import datetime
        from dateutil import tz
        from neuroconv.datainterfaces import NeuroScopeSortingInterface
        xml_path = Path(path_folder + files_basename + ".xml")
        interface = NeuroScopeSortingInterface(folder_path=Path(path_folder), xml_file_path=xml_path, verbose=False)
        metadata = interface.get_metadata()
        session_start_time = datetime(2023, 4, 4, 12, 30, 0, tzinfo=tz.gettz("US/Pacific"))
        metadata["NWBFile"].update(session_start_time=session_start_time)
        interface.run_conversion(nwbfile_path=nwbfile_path, metadata=metadata)

    return NWBFile(nwbfile_path)["units"]

spk = get_nwb(patient, sess)

n_units = len(spk)
epoch_scroll = nap.IntervalSet(start=[t_start], end=[t_end], time_units="s") # df avec debut/fin de chaque epoch
spk = spk.restrict(epoch_scroll)

spike_times = []
for i in range(n_units):
    si = spk[i]
    st = np.asarray(si.t, dtype=float)
    spike_times.append(np.sort(st))

T_total = t_end-t_start #max((st.max() if st.size else 0.0) for st in spike_times)
print(f"NWB import | Units: {n_units} | Durée totale estimée: {T_total:.2f} s")

# --- mapping unit -> group (tétrode) ---
groups_dict = spk.getby_category("group")  # dict {group_id: TsGroup}

available = sorted([float(g) for g in groups_dict.keys()])
if float(tetrode_id) not in [float(g) for g in groups_dict.keys()]:
    raise ValueError(
        f"Aucune unit trouvée pour tetrode/group={tetrode_id}. "
        f"Groupes disponibles dans ce NWB (dans l'intervalle choisi) : {available}"
    )

# indices d'unités (dans l'objet spk actuel) appartenant à la tétrode choisie
idx_keep = np.asarray(groups_dict[float(tetrode_id)].index, dtype=int)

# Filtrer spk à ces unités (garde l'ordre des indices)
spk = spk[idx_keep]

unit_keys = list(spk.index)  # <- clés réelles (ex: [10,11,12,...])

spike_times = []
for k in unit_keys:
    st = np.asarray(spk[k].t, dtype=float)
    spike_times.append(np.sort(st))

n_units = len(spike_times)

# Couleur différente par unit (autant que possible)
cmap_units = plt.get_cmap("tab20")
unit_colors = [cmap_units(i % cmap_units.N) for i in range(n_units)]

# -------------------------------------------------
# 2) Paramètres intervalle + vidéo
# -------------------------------------------------
fps = 25

# Sécurité bornes
t_start = max(0.0, float(t_start))
t_end = min(float(t_end), float(T_total))
if t_end <= t_start:
    raise ValueError(f"t_end ({t_end}) doit être > t_start ({t_start}).")

# Frames "maintenant" t dans [t_start, t_end]
frame_t = np.arange(t_start, t_end, 1.0 / fps) * speed
frame_t = frame_t[(frame_t >= t_start) & (frame_t <= t_end)]

# -------------------------------------------------
# 3) Figure
# -------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 5), dpi=120)

def setup_axes():
    ax.set_ylim(-0.5, n_units - 0.5)
    ax.set_ylabel("Unit #")
    # ax.set_xlabel("Temps dans la fenêtre (s)")
    ax.set_xlim(-window, 0.0)
    ax.set_yticks(np.arange(0, n_units, 5))
    ax.grid(True, axis="x", alpha=0.2)
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)
    ax.spines[['right', 'top', 'left', 'bottom']].set_visible(False)

setup_axes()
# ax.set_title("Raster plot défilant (intervalle sélectionné)")

# -------------------------------------------------
# 4) Update (avec extraction efficace via searchsorted)
# -------------------------------------------------
# Précompute: pour accélérer, on gardera st trié (déjà le cas).
def update(k):
    t = float(frame_t[k])
    t0 = t - window
    t1 = t

    ax.cla()
    setup_axes()
    # ax.set_title("Raster plot défilant (intervalle sélectionné)")

    lo = max(0.0, t0)

    for i, st in enumerate(spike_times):
        if st.size == 0:
            continue
        # indices des spikes dans [lo, t1] via searchsorted (plus rapide que mask)
        a = np.searchsorted(st, lo, side="left")
        b = np.searchsorted(st, t1, side="right")
        if b > a:
            x_rel = st[a:b] - t
            ax.vlines(
                x_rel,
                i - HALF_H, i + HALF_H,
                linewidth=LINE_W,
                color=unit_colors[i],
                alpha=0.95,
            )


    ax.axvline(0.0, linestyle="--", linewidth=1.0, alpha=0.7)
    ax.text(0.01, 0.98, f"t = {t:0.2f}s", transform=ax.transAxes, va="top")
    # ax.text(0.01, 0.90, f"[{t_start:.2f}, {t_end:.2f}] s", transform=ax.transAxes, va="top") # intervalle de temps

    return []

anim = FuncAnimation(fig, update, frames=len(frame_t), interval=1000 / fps, blit=False)

# -------------------------------------------------
# 5) Bande-son
# -------------------------------------------------
import wave
import subprocess

def synth_morse_audio_from_spikes(spike_times, t_start, t_end, sr=44100,
                                  click_dur=0.030, base_freq=500.0, freq_step=55.0):
    """
    Crée un son type 'morse': un petit 'TAC' (tone court) par spike au moment où il apparaît.
    - spike_times: list[np.ndarray], en secondes (absolu NWB), un array par unit
    - tonalité différente par unit via freq = base_freq + i*freq_step
    Retourne audio float32 dans [-1,1].
    """
    duration = float(t_end - t_start)
    n = int(np.ceil(duration * sr))
    audio = np.zeros(n, dtype=np.float32)

    L = max(1, int(click_dur * sr))
    tt = np.arange(L, dtype=np.float32) / sr
    # enveloppe "TAC" : attaque rapide, décroissance expo
    env = np.exp(-tt * 40.0).astype(np.float32)

    for i, st in enumerate(spike_times):
        if st.size == 0:
            continue
        f = base_freq + i * freq_step
        tone = (np.sin(2 * np.pi * f * tt) * env).astype(np.float32)

        # spikes dans l'intervalle
        st_in = st[(st >= t_start) & (st <= t_end)]
        for s in st_in:
            idx0 = int((float(s) - t_start) * sr)
            idx1 = idx0 + L
            if idx0 < 0 or idx0 >= n:
                continue
            if idx1 > n:
                tone_use = tone[: n - idx0]
                audio[idx0:n] += tone_use
            else:
                audio[idx0:idx1] += tone

    # Normaliser pour éviter clipping
    peak = np.max(np.abs(audio)) if audio.size else 0.0
    if peak > 0:
        audio = audio / (peak * 1.05)

    return audio

def write_wav(path, audio, sr=44100):
    """Écrit un WAV mono 16-bit PCM."""
    audio_i16 = np.int16(np.clip(audio, -1.0, 1.0) * 32767)
    with wave.open(str(path), "wb") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)  # 16-bit
        wf.setframerate(sr)
        wf.writeframes(audio_i16.tobytes())

def mux_audio_video_ffmpeg(video_in, audio_in, video_out):
    """
    Combine MP4 + WAV en MP4 final (AAC) via ffmpeg.
    Nécessite ffmpeg installé.
    """
    cmd = [
        "ffmpeg", "-y",
        "-i", str(video_in),
        "-i", str(audio_in),
        "-c:v", "copy",
        "-c:a", "aac",
        "-shortest",
        str(video_out),
    ]
    subprocess.run(cmd, check=True)

# -------------------------------------------------
# 6) Export
# -------------------------------------------------
output_path = f"raster_{patient}_stim{sess}_{t_start:.0f}-{t_end:.0f}s_tetrode{tetrode_id}.mp4"
# writer = FFMpegWriter(fps=fps, bitrate=4000)

# print(f"Export -> {output_path} | frames={len(frame_t)} | fps={fps}")
# anim.save(output_path, writer=writer)
# plt.close(fig)
# print("Terminé.")
from pathlib import Path

# 5) Export (vidéo temporaire) + audio + mux final
output_path = f"raster_{patient}_stim{sess}_{t_start:.0f}-{t_end:.0f}s_tt{round(tetrode_id)}.mp4"
tmp_video = Path(output_path).with_name(Path(output_path).stem + "_silent.mp4")
tmp_wav = Path(output_path).with_suffix(".wav")

writer = FFMpegWriter(fps=fps, bitrate=4000)

print(f"Export vidéo (silent) -> {tmp_video} | frames={len(frame_t)} | fps={fps}")
anim.save(str(tmp_video), writer=writer)
plt.close(fig)

# Audio "morse"
SR = 44100
audio = synth_morse_audio_from_spikes(
    spike_times=spike_times,
    t_start=t_start,
    t_end=t_end,
    sr=SR,
    click_dur=0.030,
    base_freq=500.0,
    freq_step=55.0
)
write_wav(tmp_wav, audio, sr=SR)
print(f"Audio -> {tmp_wav}")

# Mux final
mux_audio_video_ffmpeg(tmp_video, tmp_wav, output_path)
print(f"Vidéo finale avec son -> {output_path}")


NWB path: /media/aube/Aube/Spike-sorting/Data_folders/P111_SL64/P111_SL64_stim5/P111_SL64_stim5.nwb
NWB import | Units: 42 | Durée totale estimée: 20.00 s
Export vidéo (silent) -> raster_P111_SL64_stim5_0-20s_tt13_silent.mp4 | frames=500 | fps=25


/var/data/python/lib/python3.13/site-packages/matplotlib/animation.py:908: UserWarning: Animation was deleted without rendering anything. This is most likely not intended. To prevent deletion, assign the Animation to a variable, e.g. `anim`, that exists until you output the Animation using `plt.show()` or `anim.save()`.
  warnings.warn(


Audio -> raster_P111_SL64_stim5_0-20s_tt13.wav


ffmpeg version 7.1.3 Copyright (c) 2000-2025 the FFmpeg developers
  built with gcc 15.2.0 (GCC)
  configuration: --prefix=/usr --libdir=/usr/lib/x86_64-linux-gnu --optflags='-O2 -pipe -g -Wp,-D_FORTIFY_SOURCE=3 -Wp,-D_GLIBCXX_ASSERTIONS -fexceptions -fstack-protector-strong -grecord-gcc-switches -fasynchronous-unwind-tables -fstack-clash-protection -fcf-protection -fno-omit-frame-pointer -mno-omit-leaf-frame-pointer ' --extra-ldflags='-Wl,-z,relro,-z,now -Wl,--as-needed ' --disable-stripping --disable-doc --disable-static --disable-encoders --disable-decoders --enable-shared --enable-gnutls --enable-gcrypt --enable-ladspa --enable-lcms2 --enable-libaom --enable-libdav1d --enable-libmp3lame --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libjxl --enable-libopus --enable-libpulse --enable-libspeex --enable-libtheora --enable-libvorbis --enable-libvpx --enable-librsvg --enable-libwebp --enable-libxml2 --enable-libopenjpeg --enable-libsvtav1 --enable-openal --enab

Vidéo finale avec son -> raster_P111_SL64_stim5_0-20s_tt13.mp4


[out#0/mp4 @ 0x558bc6c58e40] video:1325KiB audio:148KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 1.138407%
frame=  500 fps=0.0 q=-1.0 Lsize=    1490KiB time=00:00:19.92 bitrate= 612.9kbits/s speed=35.5x    
[aac @ 0x558bc6ab1a80] Qavg: 44142.863


In [ ]:
# une seule tetrode. Pas de son ni EEG.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
import pynapple as nap

t_start, t_end = 0, 20 # en secondes (temps absolu NWB)
patient = 'P111_SL64'
sess = '5'
tetrode_id = 13.0   # <-- choisis ici (ex: 1, 4, 5, 7, 8, 13, 14 ...)


window = 4.0   # largeur de fenêtre affichée
speed = 1.0 # vitesse de défilement relativement au réel
LINE_W = 1.6      # <- épaisseur des traits (teste 1.2, 1.6, 2.0)
HALF_H = 0.45     # <- hauteur des traits (0.4 -> 0.45/0.49)

# -------------------------------------------------
# 1) Charger spikes réels
# -------------------------------------------------

import os
from pathlib import Path
from pynapple.io.interface_nwb import NWBFile

def get_nwb(patient, session, root='/media/aube/Aube/'):
    path_folder = root + 'Spike-sorting/Data_folders/'+patient+'/'+patient+'_stim'+session+'/'
    files_basename = patient+'_stim'+session
    nwbfile_path = path_folder + files_basename + ".nwb"
    print("NWB path:", nwbfile_path)

    if not os.path.exists(nwbfile_path):
        from datetime import datetime
        from dateutil import tz
        from neuroconv.datainterfaces import NeuroScopeSortingInterface
        xml_path = Path(path_folder + files_basename + ".xml")
        interface = NeuroScopeSortingInterface(folder_path=Path(path_folder), xml_file_path=xml_path, verbose=False)
        metadata = interface.get_metadata()
        session_start_time = datetime(2023, 4, 4, 12, 30, 0, tzinfo=tz.gettz("US/Pacific"))
        metadata["NWBFile"].update(session_start_time=session_start_time)
        interface.run_conversion(nwbfile_path=nwbfile_path, metadata=metadata)

    return NWBFile(nwbfile_path)["units"]

spk = get_nwb(patient, sess)

n_units = len(spk)
epoch_scroll = nap.IntervalSet(start=[t_start], end=[t_end], time_units="s") # df avec debut/fin de chaque epoch
spk = spk.restrict(epoch_scroll)

spike_times = []
for i in range(n_units):
    si = spk[i]
    st = np.asarray(si.t, dtype=float)
    spike_times.append(np.sort(st))

T_total = t_end-t_start #max((st.max() if st.size else 0.0) for st in spike_times)
print(f"NWB import | Units: {n_units} | Durée totale estimée: {T_total:.2f} s")

# --- mapping unit -> group (tétrode) ---
groups_dict = spk.getby_category("group")  # dict {group_id: TsGroup}

available = sorted([float(g) for g in groups_dict.keys()])
if float(tetrode_id) not in [float(g) for g in groups_dict.keys()]:
    raise ValueError(
        f"Aucune unit trouvée pour tetrode/group={tetrode_id}. "
        f"Groupes disponibles dans ce NWB (dans l'intervalle choisi) : {available}"
    )

# indices d'unités (dans l'objet spk actuel) appartenant à la tétrode choisie
idx_keep = np.asarray(groups_dict[float(tetrode_id)].index, dtype=int)

# Filtrer spk à ces unités (garde l'ordre des indices)
spk = spk[idx_keep]

unit_keys = list(spk.index)  # <- clés réelles (ex: [10,11,12,...])

spike_times = []
for k in unit_keys:
    st = np.asarray(spk[k].t, dtype=float)
    spike_times.append(np.sort(st))

n_units = len(spike_times)

# Couleur unique puisque 1 seule tétrode
col_tetrode = plt.get_cmap("tab20")(0)

# -------------------------------------------------
# 2) Paramètres intervalle + vidéo
# -------------------------------------------------
fps = 25

# Sécurité bornes
t_start = max(0.0, float(t_start))
t_end = min(float(t_end), float(T_total))
if t_end <= t_start:
    raise ValueError(f"t_end ({t_end}) doit être > t_start ({t_start}).")

# Frames "maintenant" t dans [t_start, t_end]
frame_t = np.arange(t_start, t_end, 1.0 / fps) * speed
frame_t = frame_t[(frame_t >= t_start) & (frame_t <= t_end)]

# -------------------------------------------------
# 3) Figure
# -------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 5), dpi=120)

def setup_axes():
    ax.set_ylim(-0.5, n_units - 0.5)
    ax.set_ylabel("Unit #")
    # ax.set_xlabel("Temps dans la fenêtre (s)")
    ax.set_xlim(-window, 0.0)
    ax.set_yticks(np.arange(0, n_units, 5))
    ax.grid(True, axis="x", alpha=0.2)
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)
    ax.spines[['right', 'top', 'left', 'bottom']].set_visible(False)

setup_axes()
# ax.set_title("Raster plot défilant (intervalle sélectionné)")

# -------------------------------------------------
# 4) Update (avec extraction efficace via searchsorted)
# -------------------------------------------------
# Précompute: pour accélérer, on gardera st trié (déjà le cas).
def update(k):
    t = float(frame_t[k])
    t0 = t - window
    t1 = t

    ax.cla()
    setup_axes()
    # ax.set_title("Raster plot défilant (intervalle sélectionné)")

    lo = max(0.0, t0)

    for i, st in enumerate(spike_times):
        if st.size == 0:
            continue
        # indices des spikes dans [lo, t1] via searchsorted (plus rapide que mask)
        a = np.searchsorted(st, lo, side="left")
        b = np.searchsorted(st, t1, side="right")
        if b > a:
            x_rel = st[a:b] - t
            ax.vlines(
                x_rel,
                i - HALF_H, i + HALF_H,
                linewidth=LINE_W,
                color=col_tetrode,
                alpha=0.95,
            )


    ax.axvline(0.0, linestyle="--", linewidth=1.0, alpha=0.7)
    ax.text(0.01, 0.98, f"t = {t:0.2f}s", transform=ax.transAxes, va="top")
    # ax.text(0.01, 0.90, f"[{t_start:.2f}, {t_end:.2f}] s", transform=ax.transAxes, va="top") # intervalle de temps

    return []

anim = FuncAnimation(fig, update, frames=len(frame_t), interval=1000 / fps, blit=False)

# -------------------------------------------------
# 5) Export
# -------------------------------------------------
output_path = f"raster_{patient}_stim{sess}_{t_start:.0f}-{t_end:.0f}s_tetrode{tetrode_id}.mp4"
writer = FFMpegWriter(fps=fps, bitrate=4000)

print(f"Export -> {output_path} | frames={len(frame_t)} | fps={fps}")
anim.save(output_path, writer=writer)
plt.close(fig)
print("Terminé.")


NWB path: /media/aube/Aube/Spike-sorting/Data_folders/P111_SL64/P111_SL64_stim5/P111_SL64_stim5.nwb
NWB import | Units: 42 | Durée totale estimée: 20.00 s
Export -> raster_P111_SL64_stim5_0-20s_tetrode13.0.mp4 | frames=500 | fps=25
Terminé.


In [ ]:
# toutes les units d'un nwb, colorées par tétrode. Pas de son.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
import pynapple as nap

t_start, t_end = 0, 20 # en secondes (temps absolu NWB)
patient = 'P111_SL64'
sess = '5'

window = 4.0   # largeur de fenêtre affichée
speed = 1.0 # vitesse de défilement relativement au réel
LINE_W = 1.6      # <- épaisseur des traits (teste 1.2, 1.6, 2.0)
HALF_H = 0.45     # <- hauteur des traits (0.4 -> 0.45/0.49)

# -------------------------------------------------
# 1) Charger spikes réels
# -------------------------------------------------

import os
from pathlib import Path
from pynapple.io.interface_nwb import NWBFile

def get_nwb(patient, session, root='/media/aube/Aube/'):
    path_folder = root + 'Spike-sorting/Data_folders/'+patient+'/'+patient+'_stim'+session+'/'
    files_basename = patient+'_stim'+session
    nwbfile_path = path_folder + files_basename + ".nwb"
    print("NWB path:", nwbfile_path)

    if not os.path.exists(nwbfile_path):
        from datetime import datetime
        from dateutil import tz
        from neuroconv.datainterfaces import NeuroScopeSortingInterface
        xml_path = Path(path_folder + files_basename + ".xml")
        interface = NeuroScopeSortingInterface(folder_path=Path(path_folder), xml_file_path=xml_path, verbose=False)
        metadata = interface.get_metadata()
        session_start_time = datetime(2023, 4, 4, 12, 30, 0, tzinfo=tz.gettz("US/Pacific"))
        metadata["NWBFile"].update(session_start_time=session_start_time)
        interface.run_conversion(nwbfile_path=nwbfile_path, metadata=metadata)

    return NWBFile(nwbfile_path)["units"]

spk = get_nwb(patient, sess)

n_units = len(spk)
epoch_scroll = nap.IntervalSet(start=[t_start], end=[t_end], time_units="s") # df avec debut/fin de chaque epoch
spk = spk.restrict(epoch_scroll)

spike_times = []
for i in range(n_units):
    si = spk[i]
    st = np.asarray(si.t, dtype=float)
    spike_times.append(np.sort(st))

T_total = t_end-t_start #max((st.max() if st.size else 0.0) for st in spike_times)
print(f"NWB import | Units: {n_units} | Durée totale estimée: {T_total:.2f} s")

# --- mapping unit -> group (tétrode) ---
groups_dict = spk.getby_category("group")  # dict {group_id: TsGroup}

unit_group = np.full(n_units, np.nan)      # unit_group[i] = group_id
for g, tg in groups_dict.items():
    # tg contient les indices d’unités (colonne "Index" dans l'affichage)
    idx = np.asarray(tg.index, dtype=int)
    unit_group[idx] = float(g)

# --- couleurs par group (via colormap) ---
unique_groups = np.array(sorted(groups_dict.keys()), dtype=float)
cmap = plt.get_cmap("tab20")  # palette catégorielle
group_color = {float(g): cmap(j % cmap.N) for j, g in enumerate(unique_groups)}

# -------------------------------------------------
# 2) Paramètres intervalle + vidéo
# -------------------------------------------------
fps = 25

# Sécurité bornes
t_start = max(0.0, float(t_start))
t_end = min(float(t_end), float(T_total))
if t_end <= t_start:
    raise ValueError(f"t_end ({t_end}) doit être > t_start ({t_start}).")

# Frames "maintenant" t dans [t_start, t_end]
frame_t = np.arange(t_start, t_end, 1.0 / fps) * speed
frame_t = frame_t[(frame_t >= t_start) & (frame_t <= t_end)]

# -------------------------------------------------
# 3) Figure
# -------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 5), dpi=120)

def setup_axes():
    ax.set_ylim(-0.5, n_units - 0.5)
    ax.set_ylabel("Unit #")
    # ax.set_xlabel("Temps dans la fenêtre (s)")
    ax.set_xlim(-window, 0.0)
    ax.set_yticks(np.arange(0, n_units, 5))
    ax.grid(True, axis="x", alpha=0.2)
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)
    ax.spines[['right', 'top', 'left', 'bottom']].set_visible(False)

setup_axes()
# ax.set_title("Raster plot défilant (intervalle sélectionné)")

# -------------------------------------------------
# 4) Update (avec extraction efficace via searchsorted)
# -------------------------------------------------
# Précompute: pour accélérer, on gardera st trié (déjà le cas).
def update(k):
    t = float(frame_t[k])
    t0 = t - window
    t1 = t

    ax.cla()
    setup_axes()
    # ax.set_title("Raster plot défilant (intervalle sélectionné)")

    lo = max(0.0, t0)

    for i, st in enumerate(spike_times):
        if st.size == 0:
            continue
        # indices des spikes dans [lo, t1] via searchsorted (plus rapide que mask)
        a = np.searchsorted(st, lo, side="left")
        b = np.searchsorted(st, t1, side="right")
        if b > a:
            x_rel = st[a:b] - t

            g = float(unit_group[i]) if not np.isnan(unit_group[i]) else None
            col = group_color[g] if g in group_color else "k"
            ax.vlines(
                x_rel,
                i - HALF_H, i + HALF_H,
                linewidth=LINE_W,
                color=col,
                alpha=0.95,
            )
            # ax.vlines(x_rel, i - 0.4, i + 0.4, linewidth=0.7)

    ax.axvline(0.0, linestyle="--", linewidth=1.0, alpha=0.7)
    ax.text(0.01, 0.98, f"t = {t:0.2f}s", transform=ax.transAxes, va="top")
    # ax.text(0.01, 0.90, f"[{t_start:.2f}, {t_end:.2f}] s", transform=ax.transAxes, va="top") # intervalle de temps

    return []

anim = FuncAnimation(fig, update, frames=len(frame_t), interval=1000 / fps, blit=False)

# -------------------------------------------------
# 5) Export
# -------------------------------------------------
output_path = f"raster_{patient}_stim{sess}_{t_start:.0f}-{t_end:.0f}s.mp4"
writer = FFMpegWriter(fps=fps, bitrate=4000)

print(f"Export -> {output_path} | frames={len(frame_t)} | fps={fps}")
anim.save(output_path, writer=writer)
plt.close(fig)
print("Terminé.")


NWB path: /media/aube/Aube/Spike-sorting/Data_folders/P111_SL64/P111_SL64_stim5/P111_SL64_stim5.nwb
NWB import | Units: 42 | Durée totale estimée: 20.00 s
Export -> raster_P111_SL64_stim5_0-20s.mp4 | frames=500 | fps=25
Terminé.
